# OCR a ticker's filings

## 1 · Parameters — the only cell you edit

In [2]:
# ═══════════════════════════════════════════════════════════════════════════════════
#  HOSE_GAS — what is true of THIS ticker, MEASURED from disk 2026-09-07
# ═══════════════════════════════════════════════════════════════════════════════════
#  template `corp` (RESOLVED by CafeF's fingerprint, over the network — `TPX-1`)
#  61 quarter(s) filed  ·  16 complete  ·  **123 `pdf` cells of 183**  ·  **60 cells OPEN**
#
#      balance_sheet     43 / 61 `pdf`    18 open
#      income_statement  54 / 61 `pdf`     7 open
#      cash_flow         26 / 61 `pdf`    35 open   <- the weak statement, and by a wide margin
#
#      45 quarters carry at least one open cell  ·  **0 SETTLED cells**  ·  **0 span operands**
#      `documents()`: 61 filings, 59 consolidated / 2 parent-only, 31 cumulative
#      first_report 2011-Q4 — where the CONTIGUOUS filing chain starts, read from the PDF
#      FILES on disk (168 files, 847 MB): 0 filing CafeF advertises is missing its PDF here.
#
#  ⚠️ **THE 2026-09-06 HEADER SAID "135 CELLS STILL OPEN". IT IS 60, AND 60 IS 183 − 123.**
#     Re-measured 2026-09-07 with the same two calls §3 makes (`plan_batch` +
#     `builder._existing`). The 123/183 in that header was right and the 135 beside it cannot
#     be reconciled with it; a count that disagrees with its own neighbour is the kind this
#     file exists to correct rather than carry (§5 rule 2 — an unmeasured number is not a
#     measurement).
#
#  ⚠️ **BOOTSTRAPPED ON A T4 ON 2026-09-05, AND THE GUARD WAS LIFTED TO DO IT.** Run folder
#     `20260905-204946__hose_gas__pdf_ocr`: 61 documents, 0 engine errors, stack
#     `88df8ef02c08`, 0 pin violations, 127 of 183 statements accepted (69.4 %). The ticker
#     had NO statement CSV, so `seed_history` could build no band for any quarter — 183 of
#     183 EMPTY — and `pdf_ocr_merge` refuses an empty-band statement, which is `BND-1`'s
#     loop: nothing written, so the band stays empty. It was merged with
#     `force_empty_band=True`, one period per call, oldest first, `force_differs=False`.
#     ⚠️ **SO NONE OF THE 123 ROWS ON DISK PASSED A MAGNITUDE GUARD.** What replaced it is
#     below, and it is arithmetic.
#
#  ⚠️ **THE EMPTY-BAND COUNT IS 5 OF 60, NOT "3, ONE PER STATEMENT, ALL Q4-2008".** Measured
#     2026-09-07 the way refusal 2 measures it — `seed_history(before=<period>)` read at the
#     document's OWN entity (`SAN-1` bands per entity), per open cell:
#
#         2008-Q4  balance_sheet     consolidated=False    <- a PARENT-COMPANY filing
#         2008-Q4  cash_flow         consolidated=False
#         2010-Q4  balance_sheet     consolidated=True
#         2010-Q4  cash_flow         consolidated=True
#         2011-Q4  balance_sheet     consolidated=True
#
#     **55 of the 60 open cells HAVE a band.** Q4-2008's income statement is already `pdf`,
#     so it is not open at all — which is why the old "one per statement" reading was wrong.
#     2010-Q4 and 2011-Q4 are empty for a reason that is not the bootstrap: GAS filed only
#     ANNUAL reports in 2008-2011, Q4-2009's balance sheet is `consolidated=False`, and a
#     band is per entity — so the `True` band before Q4-2010 has nothing in it by
#     construction. ⚠️ **AND THE WORKER'S BAND IS A SNAPSHOT**: `history_sizes` is computed
#     on the T4 from the payload's CSVs, so writing 2010-Q4 in this run cannot give 2011-Q4
#     a band in the same run. Ordering the merge does not fix these five; only a later run does.
#
#  ⚠️ **BUT THE GUARD IS NOT WHAT DECIDES THOSE FIVE ANY MORE — the all-three gate is.**
#     Since 2026-09-06 `merge_batch` passes `force_empty_band=force_empty_band or period in
#     complete_periods(folder)`, so a quarter whose FILING produced all three statements is
#     written band or no band, on this KAGGLE path exactly as on the local one. What
#     `FORCE_EMPTY_BAND` still decides here is a filing that produced TWO of three — which is
#     a judgement about that filing, and stays False.
#
#  ⚠️ **Q2-2021 CASH FLOW IS `missing` ON PURPOSE, AND 2021-Q2 IS IN THIS RUN'S 45.** It is
#     not an OCR gap and the last cascade did not fix it — the CLOSING balance is misread:
#
#         opening  5,237,246,729,402   <- corroborated to the dong by Q4-2021's own opening,
#                                         read from a DIFFERENT filing
#         net        712,235,211,027
#         fx          -1,921,300,763
#         => closing 5,947,560,639,666   what the identity requires
#         read       1,997,560,636,666   what `onnx@400` returned  (note the shared tail)
#
#     ⚠️ **NOTHING IN THE MERGE WILL STOP IT BEING WRITTEN THIS TIME.** The cell is `missing`,
#     so there is no `pdf` row to raise DIFFERS against, and the misread is the same order of
#     magnitude as the truth, so `sane` passes it. Last time it was held through the merge's
#     `reports` filter, NEVER by editing the artefact. **Decide before §9 runs** — read the
#     Q2-2021 cash flow in §7 first, and if it reads 1,997,560,636,666 again, merge in two
#     calls (`MERGE_REPORTS` without `cash_flow`, then a scoped `merge_run` for `cash_flow`)
#     rather than letting one sweep write it. To repair it properly, decide against the
#     FILING — not by preferring a newer run.
#
#  ⚠️ **TWO MORE CASH FLOWS DO NOT CLOSE EXACTLY, AND THE SCREENS PASS THEM BY DESIGN.**
#     `statement_screens.REL_TOL` is 5e-3, so these sit inside it:
#         Q4-2022  out by exactly **1,000,000,000** on 10.5 tn  (9.5e-5)
#         Q2-2023  out by 3,013,219,678 on 12.5 tn              (2.4e-4)
#     A round 1 bn is what one misread digit looks like; the other is the size of an unmapped
#     line. `GTL-1` learned this on a balance sheet — a tolerance wide enough to be useful is
#     wide enough to pass a wrong digit — and neither has been adjudicated against the filing.
#
#  ⚠️ **Q2-2022 CARRIES NO ROW AT ALL, AND THE 2026-09-06 HEADER GAVE THE WRONG REASON.** It
#     said the balance sheet and income statement were refused with *"no such statement on any
#     page of this filing"* **on every layer**. The run folder says otherwise:
#
#         balance_sheet     onnx@200        no such statement on any page of this filing
#                           onnx@200+title  reconcile: only 1 rows parsed
#         income_statement  onnx@200        no such statement on any page of this filing
#                           onnx@200+title  reconcile: only 8 rows parsed
#         cash_flow         onnx@200        reconcile: no closing cash balance
#
#     **That mixture is exactly why this ticker has 0 SETTLED cells**: `settled_absences`
#     needs EVERY recorded reason to be the permanent one, and the second layer's is a
#     reconcile refusal. So Q2-2022 is `open`, it is in the 45, and a re-run may still win it —
#     which is the correct reading, not the one the old header implied. `SET-2` cuts the other
#     way too: that first reason is a verdict on the PAGE CLASSIFIER wearing the words of one
#     on the document.
#
#  ⚠️ **9 QUARTERS HAVE NO FILING AT ALL AND `missing` IS CORRECT AND PERMANENT** (§5 rule
#     24): Q1-Q3 of 2009, 2010 and 2011. GAS filed ONE audited annual report in each of
#     2008-2011 and its first QUARTERLY report is 2012-Q1 — 11-13 documents a year after that.
#     That is why the chain starts at 2011-Q4 and 61 filed quarters span 2008-Q4 … 2026-Q1
#     rather than 70.
#
#  ⚠️ **FOUR Q4 INCOME STATEMENTS ARE ON DISK CARRYING `months = 12`, AND THAT IS CORRECT.**
#     Q4-2008..Q4-2011 are annual reports whose Q1..Q3 were never filed, so `FY − (Q1+Q2+Q3)`
#     has no operands and never will; the merge keeps such a row labelled rather than dropping
#     it. ⚠️ **Any TTM or ratio built from those four would count a YEAR as a QUARTER — the
#     span column is the only thing that says so.** Q4-2009's balance sheet is
#     `consolidated=False`: a PARENT-COMPANY filing, reached through `ALLOW_PARENT`.
#
#  ⚠️ **`CRP-1`: NOTHING FROM THIS TICKER MAY BE QUOTED AS A FUNDAMENTAL.** GAS files on the
#     `corp` chart, where `C_LIABILITIES` does not map — so `reconcile` tests
#     `assets == resources`, true by construction on any page that reads both totals, and
#     never `A = L + E`. `SEC-1`'s section sums and `GTL-1`'s tolerance are real gates where
#     there was none; that is not the same thing as a checked balance sheet.
#
#  ⚠️ **`TPX-1`: `templates.csv` HOLDS ACB, BID AND VCB AND NOT GAS**, so `corp` is resolved by
#     a NETWORK call to CafeF's own fingerprint on every run — confirmed today
#     (`template_how = detect_template`). ⚠️ **AND GAS IS IN NEITHER FINANCIALS REGISTER**:
#     `CAFEF_FINANCIALS_TICKERS` holds 4 names (VCB, ACB, BID, VIC) and config.json's
#     `raw/cafef_financials` partitions hold the same 4 — so its Dagster asset cannot be
#     materialised and its statements feed no silver ingest: parsing it changes no table.
#     ⚠️ **The 2026-09-06 header said "not `orchestration/config.json`" and that is loose** —
#     GAS *is* in that file, under `partitions.unified` (37 entries), which is a DIFFERENT
#     asset and changes nothing here. This notebook needs neither register; the Dagster path does.
#
#  ⚠️ **WHAT THE 45 OPEN DOCUMENTS COST LAST TIME — measured, not estimated.** The bootstrap's
#     own `documents/*.json` carry `seconds`: these same 45 filings summed **158.5 min of
#     document time** (mean 3.5 min, slowest 2016-Q2 at 7.6). ⚠️ **That is DOCUMENT time and
#     not wall clock** — the bootstrap's 61 documents summed 204.8 min against a run reported
#     at 128.4, on a worker `metadata.json` records as **2 × Tesla T4**. Budget on the
#     document sum and read the ratio as the machine's, not as a speedup you can quote
#     (`PDF_OCR.md` §3: to compare two machines, INTERLEAVE the runs).
#
#  ⚠️ **KAGGLE ALLOWS TWO BATCH GPU SESSIONS AND A THIRD PUSH IS REJECTED**, not queued —
#     `Maximum batch GPU session count of 2 reached`, measured 2026-09-05 while VIC was also
#     running. `python -m kgpu` has no verb for "wait on a kernel someone else pushed";
#     `runner.wait(cfg)` + `runner.pull(cfg)` on a config built with the SAME parameters is
#     how that was done, and the job name is what has to match.
# ═══════════════════════════════════════════════════════════════════════════════════

# ── PARAMETERS — the only cell you edit ───────────────────────────────
ENVIRONMENT = "KAGGLE"       # "LOCAL" = parse here | "KAGGLE" = ship it to a T4
EXCHANGE    = "HOSE"         # HOSE | HNX | UPCOM
SYMBOL      = "GAS"          # ticker, as CafeF files it

# WHICH QUARTERS — A LIST, AND NOTHING ELSE. Each entry is YYYY-QQ; "2026-Q4" and the
# zero-padded "2026-04" are the same quarter, folded once at the edge.
#   []                     ->  EVERY quarter this ticker files  (⚠️ ~70 documents, hours).
#                              ⚠️ Safe on this 4 GiB card ONLY because `ISOLATE_DOCUMENTS`
#                              is on; the same list in one process died at document 4
#                              (`GPU-1`). `ONLY_MISSING` below narrows it to the gap.
#   ["2014-Q4", "2015-03"] ->  exactly these quarters, and nothing else.
# ⚠️ The repo-native "Q3-2014" is REFUSED rather than quietly accepted: a typo has to report
#    itself as a typo, not as a quarter CafeF does not file. A STRING is refused for the same
#    reason — a bare "2014-Q4" with the brackets forgotten included; §2 has the measurement.
QUARTERS = []   # ⚠️ THE DEFAULT CHANGED 2026-09-03 AND IT IS THE EXPENSIVE DIRECTION. This
                # read "OUTSTANDING", which resolved to the GAP and raised when there was none;
                # `[]` is every quarter the ticker files — ~70 documents and hours — on a
                # notebook somebody just pressed run on. §2 and §3 both print which it is and
                # how many documents, before anything is spent.
                #   the old default back:  ONLY_MISSING = True
                #   one quarter:           QUARTERS = ["2014-Q4"]

# ⚠️ NARROW AN EMPTY `QUARTERS` TO WHAT IS STILL MISSING — read ONLY when the list is empty,
#    and it is what the retired "OUTSTANDING" sentinel became.
#   False -> every quarter the ticker files, which is what `[]` says above.
#   True  -> exactly the quarters §3 finds still `missing` AND still winnable, plus the span
#            operands they need. Resolved from the three statement CSVs and the PDF index,
#            printed before anything is spent, and it RAISES rather than falling through to
#            "every quarter" when there is nothing left to do.
ONLY_MISSING = True   # ⚠️ TRUE SINCE THE 2026-09-05 BOOTSTRAP: the ticker now HAS statement
                      #    CSVs, so parse the GAP and not the ticker. Re-measured 2026-09-07:
                      #    **45 quarters of 61 still carry an open cell**, and re-opening the
                      #    16 complete ones would pay the cascade to return what disk holds.

# ⚠️ **`open` IS NOT `WINNABLE`, AND §3 CANNOT TELL YOU WHICH — so read this before setting
#    ONLY_MISSING = True on a ticker you have already run.** `settled_absences` records one
#    reason and one only: `no such statement on any page of this filing`, which is a verdict
#    on the DOCUMENT and therefore permanent. Every OTHER refusal — a total that will not
#    balance, an identity that does not close, a magnitude the guard rejected — is reported
#    as `open — a re-run could still win it`, because a later layer or a fixed anchor could
#    in principle overturn it. Re-running one costs the FULL cascade to return the same word.
# ⚠️ **AND THIS TICKER HAS 0 SETTLED CELLS, WHICH IS SILENCE AND NOT A CLEAN BILL** (§5 rule
#    2). All 60 open cells read `open` here; Q2-2022's entry above shows what that can mean —
#    a first layer saying the statement is not in the document and a second saying it parsed
#    too few rows. So `45 quarters` is an upper bound on what is winnable, never a forecast
#    of what will be won.
#
# ⚠️ **AND WHEN ONE COMES BACK `absent` TWICE, THE FIRST THING TO CHECK IS THE PDF INDEX,
#    NOT THE LAYERS.** `documents()` returns ONE filing per period and a quarter can have
#    several. Measured 2026-09-04 on TCB's Q2-2019: its closing cash balance is printed under
#    the company's round stamp in the AUDITED consolidated filing, so the recogniser returns
#    a different wrong figure at 200, 300, 400+pad6, 500 and 600 dpi and never the printed
#    one — and the REVIEWED consolidated filing of the same quarter is a different scan that
#    reads the whole tail cleanly at layer 1. *No OCR configuration can read this figure* was
#    measured, true, and written up as *this quarter cannot be parsed*, which is a claim about
#    a different thing. `_alternate_retry` (`ALT-1`) now tries the others automatically; §8
#    prints which filing each recovered statement came from.
#    So: read §8's `absent_reasons`, check the index for a second filing, and write whatever
#    you settle down where a reader meets it BEFORE spending the cascade again — this comment
#    or the per-ticker notebook.
#
# ⚠️ **AND A `SETTLED` CELL IS NOT PROOF EITHER — `SET-2`, measured 2026-09-04.** The one
#    reason `settled_absences` treats as PERMANENT, `no such statement on any page of this
#    filing`, is a verdict on the PAGE CLASSIFIER and reads as one on the document. TCB's
#    Q1-2017 and Q3-2017 print the notes title AND the notes form code on the cash flow's
#    FIRST page, so no cash-flow page is found and both were recorded as filings containing
#    no cash flow — page 8 of Q1-2017 prints "LƯU CHUYỂN TIỀN THUẦN TỪ HOẠT ĐỘNG KINH
#    DOANH" over 67 figures. §3 DROPS such a cell before any OCR, so re-trying one means
#    naming its quarter in QUARTERS explicitly.

# ⚠️ What to do about a quarter ALREADY on disk:
#   False -> FILL THE GAPS. One reading `pdf` in all three statements is dropped before any OCR
#            (and before it is uploaded); a figure that DIFFERS is never written over it.
#   True  -> re-parse every selected quarter and let the result replace what disk holds.
#   ⚠️ To replace ONE wrong row use REPAIR below, never this — see the note there.
# ⚠️ TRUE IS REQUIRED BY `SPAN_OPERANDS`: a span operand is BY DEFINITION a quarter already
#    reading `pdf`, so with False it is dropped before any OCR and the Q4 it unblocks stays
#    unwritable. §2 refuses SPAN_OPERANDS without it.
OVERWRITE = False   # ⚠️ FALSE, AND IT IS MEASURED RATHER THAN PREFERRED: `plan_batch` reports
                    #    **0 span operands** for GAS — asked BOTH ways on 2026-09-07,
                    #    `span_operands=False` and `=True`, and the answer is 45 quarters /
                    #    0 operands either way. So there is no operand to protect and False is
                    #    strictly safer: a quarter complete in all three is dropped before any
                    #    OCR, and a DIFFERS is refused.

# UPSERT the accepted statements into raw_data/.../statements/*.csv, through `pdf_ocr_merge`:
# it BACKS THE THREE CSVs UP FIRST, prints every changed cell, and refuses four things it
# cannot judge — a statement whose `sane` band was empty, a figure that DIFFERS from a good
# `pdf` row, a cumulative income statement whose priors it cannot subtract, and ⚠️ a document
# any of whose layers RAISED (`VCR-1`: an exception measures the MACHINE, not the filing, so
# whatever won the cascade won by default).
# ⚠️ OFF, AND §9 DOES THE UPSERT, for the reason that survives OVERWRITE being False:
#    `merge_run` PLANS THE WHOLE FOLDER AGAINST DISK AND WRITES AFTERWARDS, so one blanket
#    call would decide a Q4 while the Q3 span it depends on is still whatever disk held when
#    the call started. §9 merges one period at a time, oldest first, which is the only shape in
#    which a span operand reaches the quarter it exists to unblock.
# ⚠️ AND ON THIS TICKER IT IS ALSO THE Q2-2021 HOLD: the pull's blanket call takes no
#    `reports` filter you can scope per period, and §9's does. See the header.
# ⚠️ OFF NO LONGER MEANS "THE CSVs ARE LEFT ALONE" — §9 WRITES BY DEFAULT since 2026-09-04
#    (`MERGE_APPLY = True` below). What this flag decides is only WHICH path does the upsert.
MERGE_INTO_CSV = False

# ⚠️ WRITE A STATEMENT WHOSE `sane` BAND WAS EMPTY — it lifts a real guard (`BND-1`).
#   True  -> write it anyway.
#   False -> keep the guard.
# ⚠️ **IT NO LONGER DECIDES WHETHER A NEW TICKER CAN START, AND THAT CHANGED 2026-09-06.** A
#    quarter whose filing produced ALL THREE statements is written band or no band, by both
#    writers, because refusing it was `BND-1`'s loop rather than a guard — see MERGE_EACH.
#    What this flag still governs is everything that gate does NOT cover: a filing that
#    produced two statements of three, which is a judgement about THAT filing and stays the
#    operator's.
FORCE_EMPTY_BAND = False   # ⚠️ FALSE, AND THE COST IS QUANTIFIED — 5 CELLS OF 60, AND ONLY
                           #    IF THEIR FILING PRODUCES FEWER THAN THREE STATEMENTS.
                           #    Re-measured 2026-09-07 over the 60 open cells the way refusal
                           #    2 measures it: **55 have a band, 5 do not** (2008-Q4 bs+cf at
                           #    entity False, 2010-Q4 bs+cf, 2011-Q4 bs — the header lists
                           #    them and says why). The all-three gate covers any of those
                           #    five whose filing comes back complete, so True would buy at
                           #    most the two-of-three cases among them — and would lift the
                           #    guard for all 60. On a ticker whose 123 rows already passed no
                           #    magnitude guard, that is the wrong direction: this is the only
                           #    magnitude guard these rows will ever meet.

# ⚠️ THE ONNX-ONLY CASCADE — 53 layers of 55, and it is about REPRODUCING, not about speed.
#   True  -> drop `tesseract@200` and `tesseract@400+relax`.
#   False -> the full cascade as shipped.
# ⚠️ **THE REASON IS PROVENANCE, AND THE OLD REASON IS STALE.** This comment used to say
#    `tesseract@200` DOES NOT EXIST on a Kaggle worker (`TSS-1`). ⚠️ **That is no longer true
#    and this ticker's own run folder is the evidence**: `20260905-204946`'s `metadata.json`
#    records `tesseract.ready = True`, our `vie.traineddata` shipped in the payload, 12,435,550
#    bytes, `matches_pin = True`. The payload carries the model now. What stands is the
#    provenance argument: **all 123 `pdf` rows on disk were written by the 53-layer cascade**,
#    so a run under the full 55 is a DIFFERENT PROCEDURE and reports the difference as DIFFERS.
#    Measured on BSR Q3-2019: `tesseract@200` read 361,884,738 where the Kaggle `onnx@300+tail`
#    row reads 361,884,738,267.
ONNX_ONLY = True

# ⚠️ PULL IN THE QUARTERS A CUMULATIVE Q4 NEEDS AS OPERANDS (`QUARTERS = []` with
#    ONLY_MISSING = True only — the other two modes already name every quarter they are going
#    to open, so there is nothing left for this to add).
#   A Q4 income statement is the YEAR, and the standalone quarter is FY − (Q1+Q2+Q3). The
#   merge will only subtract a prior whose span is a KNOWN three months, and most of the
#   corpus predates the `months` column — so the priors read `unrecorded`, a blank is NOT 3
#   (§5 rule 2), and the Q4 is refused however well it parsed.
# ⚠️ MEASURED: CTG carried SEVEN such Q4 income statements on 2026-09-02, every one of them
#    parsed and none of them writable, blocked by a blank column in ANOTHER ROW. Re-parsing a
#    prior moves no figure — an unchanged reading goes through the merge's `fills_span`
#    branch, which writes the span and nothing else.
SPAN_OPERANDS = False   # ⚠️ FALSE: 0 operands measured 2026-09-07 (see OVERWRITE above), and
                        #    §2 raises if this is True while OVERWRITE is False. ⚠️ 9 of the 45
                        #    open quarters ARE Q4s and 31 of the 61 filings are cumulative —
                        #    the four 2008-2011 annuals are the `months = 12` rows in the
                        #    header, and no run can ever split them.

# ⚠️ ONE PROCESS PER DOCUMENT — what makes a WHOLE-TICKER run possible on a 4 GiB card.
#   True  -> `pdf_ocr_batch.run_batch`: a fresh process per filing, and it waits for the card
#            to have VRAM_FLOOR_MB free before each one.
#   False -> `pdf_ocr_job.run` parses every filing in THIS process. Right for one quarter.
# ⚠️ MEASURED 2026-09-02: 18 documents in one process cleared three filings and then every
#    `onnx@*` layer raised `CUDA failure 2: out of memory` — 294 of them — and the cascade
#    went on and reported `pdf` for statements it had been unable to read. The same 25
#    documents, one process each, ran with **0 engine errors**. It changes no semantics:
#    `seed_history` re-seeds `sane` from DISK per document and the page cache is per filing.
# ⚠️ The cost is model load, ~10-20 s per document.
ISOLATE_DOCUMENTS = True
VRAM_FLOOR_MB = 2600     # free VRAM one document wants before it starts; a filing peaked at 2.9-3.2 GiB
SHOW_ABSENT_ROWS = True  # §8 prints the rows behind a REFUSED statement — the cause, not the symptom

TEMPLATE     = None      # None = RESOLVE it (templates.csv, then CafeF's fingerprint). ⚠️ Never defaulted to "bank".
                         # ⚠️ Resolves to `corp` by the NETWORK route for GAS — `templates.csv`
                         # names only ACB, BID, VCB (`TPX-1`). The bootstrap STATED it
                         # (`template_how = override`); left None here so the run records
                         # which route answered, which is a different claim.
ALLOW_PARENT = True      # fall back to the STANDALONE filing where no consolidated one exists.
                         # ⚠️ LOAD-BEARING HERE: 2 of the 61 filings are parent-only, and both
                         # of Q4-2008's open cells sit at entity `False` because of it.
PERIODS      = None      # the repo-native form, e.g. ["Q3-2014"]. Optional, and INTERSECTS with QUARTERS.
LAYERS       = None      # None = the cascade ONNX_ONLY selects, in cascade order
COMPARE      = True      # score every parsed cell against the statement CSV already on disk
NOTES        = ""        # free text into the run folder; blank writes a sensible default

# ── THE MERGE — as the run goes (§6), then the sweep (§9) ────────────────────────────
# ⚠️ Merging period by period is not a style choice: `merge_run` plans against disk and writes
#    afterwards, so a span recorded for one quarter reaches the NEXT quarter's planner only in
#    the following call. That is the dependency a span operand needs.

# ⚠️ WRITE EACH QUARTER THE MOMENT ITS FILING HAS PRODUCED ALL THREE STATEMENTS, instead of
#    waiting for §9. ⚠️ **LOCAL + ISOLATE_DOCUMENTS ONLY** — on KAGGLE the worker's data root
#    is a payload that dies with the kernel (`pdf_ocr_job.run` refuses to merge there at all),
#    so the write is the pull's; on the one-process path it is `MERGE_INTO_CSV` above.
# ⚠️ **SO ON THIS RUN IT DOES NOTHING, AND IT IS LEFT TRUE ON PURPOSE**: ENVIRONMENT is
#    "KAGGLE", so every statement this run produces reaches disk through §9 and nowhere else.
#    §2 prints which writer is live before anything is spent — read that line.
# ⚠️ **THE GATE IS THE FILING, NOT THE STATEMENT.** A document that accepted two of three is
#    HELD — and named in the log as it happens — for §9, where you are reading the refusals.
#    The three CSVs of a quarter move together or they do not move.
# ⚠️ **AND IT LIFTS EXACTLY ONE GUARD, FOR EXACTLY THAT GATE**: a complete quarter is written
#    band or no band, because refusing on an empty band closes `BND-1`'s loop rather than
#    guarding anything. Each such row is PRINTED and RECORDED as unguarded — in the run
#    folder's `merge` block, and again in §10. ⚠️ **THOSE ROWS PASSED NO MAGNITUDE GUARD.
#    SCREEN THEM BY ARITHMETIC BEFORE QUOTING ANY OF THEM** — two statements agreeing on one
#    figure, a printed subtotal closing. On GAS that is the whole 123-row history, not an edge
#    case.
MERGE_EACH = True

MERGE_TWO_PASS = True
MERGE_REPORTS = None     # ⚠️ WHICH STATEMENTS §9 MAY WRITE. None = all three.
                         # ⚠️ **THIS IS THE Q2-2021 LEVER, AND IT IS ALL-OR-NOTHING ACROSS THE
                         # RUN** — it filters by REPORT, not by (period, report). If §7 shows
                         # Q2-2021's cash flow reading 1,997,560,636,666 again, do NOT let one
                         # sweep write everything: run §9 with
                         # ["balance_sheet", "income_statement"] first, then a scoped
                         # `merge_run` for `cash_flow` with its own `periods` filter.
MERGE_APPLY   = True     # ⚠️ THE DEFAULT SINCE 2026-09-04, AND IT IS WHAT MAKES THIS
                         # NOTEBOOK WRITE. It was False, so a run that parsed perfectly
                         # ended in a PLAN and the three statement CSVs were never opened.
                         # ⚠️ MEASURED ON HOSE_FPT, 2026-09-04: a 185-minute T4 round trip
                         #    over 71 filings accepted 128 of 213 statements, §9 planned
                         #    **96 WRITEs**, and **0** of them reached disk. Two knobs had
                         #    to be flipped by hand afterwards to finish a job the machine
                         #    had already done — `BND-1`'s loop wearing a second face: the
                         #    work is on disk, the CSV is not, and a green run says nothing
                         #    about which.
                         #   False -> PLAN ONLY. Right when you are about to REPAIR a row,
                         #            or want to read the refusals before spending disk.
                         # ⚠️ WHAT MAKES AN AUTOMATIC WRITE DEFENSIBLE IS THE REFUSALS, NOT
                         # THE EXTRA COMMAND. §9 passes `force_differs=False`, so a figure
                         # that DIFFERS from a good `pdf` row on disk is STILL refused however
                         # `OVERWRITE` is set — and the other three refusals stand untouched.
                         # A backup of the three CSVs is taken by the first call that writes
                         # anything, and every changed cell is printed. `REPAIR` in §11 is
                         # still the only way past DIFFERS, and it is still opt-in and scoped.
                         # ⚠️ AND A DRY RUN UNDERSTATES A TWO-PASS WRITE, BY CONSTRUCTION:
                         # with nothing written, a later period is planned against the span
                         # the earlier one has not recorded yet, and reports the refusal it
                         # always would. GAS measured it — **the dry run said 101 and the
                         # merge wrote 123.** Read §9's applied count, never its plan.

# ⚠️ REPAIR — REPLACE A `pdf` ROW THAT IS ALREADY ON DISK AND WRONG. Name the exact
#    (quarter, statement) pairs; anything not named keeps the DIFFERS refusal. The quarter is
#    the REPO-NATIVE form here:   REPAIR = [("Q3-2019", "income_statement")]
# ⚠️ `OVERWRITE = True` IS THE WRONG TOOL FOR THIS, AND THE REASON IS MEASURED. It lifts DIFFERS
#    for every statement of every quarter in the run — and a `pdf_ocr_job` run is NOT the run
#    that wrote those rows: its `sane` band is rebuilt from disk where a full `build()`
#    accumulates one as it goes, so the two escalate DIFFERENTLY and the seeded run can win on
#    an EARLIER, POORER layer. On ACB 2026-08-30 it would have replaced a 33-item balance sheet
#    with a 19-item one while repairing another statement, reporting only "DIFFERS in N columns".
# ⚠️ Read the DIFFERS report in section 7 first, and decide against the FILING (a printed
#    subtotal, the next quarter's comparative column) — never by preferring the newer run.
# ⚠️ **EMPTY, AND Q2-2021 IS NOT A REPAIR**: its cash flow is `missing`, not a wrong `pdf` row,
#    so REPAIR has nothing to lift. What it needs is the §9 hold above.
REPAIR = []
REPAIR_APPLY = False     # False = print the plan and change nothing. True once you agree.

EXECUTE  = True          # False = resolve and print the plan, spend nothing
REHEARSE = True          # KAGGLE only: the worker side, locally, no quota (~60 s)

## 2 · Setup — validate the parameters, find the repo

In [4]:
# ── SETUP — validate the parameters and find the repo ─────────────────────────
# ⚠️ Checked HERE, before a payload is built or a page is rendered: every one of these is a
# mistake that would otherwise surface hours later, or as a spent Kaggle round trip.
import os
import sys
from pathlib import Path

ENVIRONMENT = str(ENVIRONMENT).upper()
EXCHANGE = str(EXCHANGE).upper()
SYMBOL = str(SYMBOL).upper()
if ENVIRONMENT not in ("LOCAL", "KAGGLE"):
    raise ValueError(f"ENVIRONMENT must be 'LOCAL' or 'KAGGLE', not {ENVIRONMENT!r}")
if EXCHANGE not in ("HOSE", "HNX", "UPCOM"):
    raise ValueError(f"EXCHANGE must be HOSE, HNX or UPCOM, not {EXCHANGE!r}")

REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src" / "kaggle_gpu").is_dir()), None)
if REPO is None:
    raise RuntimeError(f"no src/kaggle_gpu at or above {Path.cwd()} — open this notebook "
                       f"from inside the repo.")
# ⚠️ `kgpu` stages the payload and talks to the Kaggle client relative to the CWD, so the
# notebook anchors itself the way a shell would. LOCAL does not need it and gets it anyway:
# one behaviour, printed, beats two that differ by a mode.
os.chdir(REPO / "src" / "kaggle_gpu")
for _p in (REPO / "src", REPO / "src" / "kaggle_gpu"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

# ⚠️ A LONG-LIVED KERNEL PINS THE REPO TO THE COMMIT IT FIRST IMPORTED — `import` is a no-op
# once a module is in `sys.modules`, so re-running this notebook after the repo moves underneath
# it runs the OLD code. The loud form is an AttributeError; ⚠️ the silent form is an OCR run
# executing a previous commit's parser while `metadata.json` records HEAD's hash — a run folder
# that names code it did not run. So the repo's OWN packages are dropped here and re-imported
# from disk on every pass; third-party ones (torch, onnxruntime) are left alone, they do not
# move. ⚠️ It re-imports, so run this notebook TOP TO BOTTOM.
_OURS = ("kgpu", "utils", "web_scraper")
_RELOADED = [_n for _n in list(sys.modules) if _n.split(".")[0] in _OURS]
for _n in _RELOADED:
    del sys.modules[_n]

from utils import progress                          # noqa: E402
from web_scraper import pdf_ocr_job as job          # noqa: E402

# ⚠️ FOLDED ONCE, HERE. "2026-04" and "2026-Q4" are one quarter, and the job name, the payload
# directory and the Kaggle kernel slug are all derived from this list — two spellings that
# reached those would be two runs racing for one slug. An EMPTY list folds to `None`, which is
# `plan()`'s own contract for "every quarter this ticker files".
# ⚠️ A LIST, AND NOTHING ELSE (2026-09-03). QUARTERS used to take the strings "ALL" and
# "OUTSTANDING" beside the list, and TWO TYPES IN ONE PARAMETER COST THREE MEASURED READINGS,
# every one of which reported the wrong mistake:
#   QUARTERS = ""          an empty string is FALSY, so it fell past the sentinel test into
#                          `canonical_quarters`, which reads empty as `None` — it opened EVERY
#                          quarter this ticker files, silently, and printed "ALL".
#   QUARTERS = "  "        strips to "" and falls the same way, except `canonical_quarters`
#                          then iterates the string CHARACTER BY CHARACTER: `' ' is not a
#                          quarter`, an error about the QUARTER FORM for a mistake in the MODE.
#   QUARTERS = "2014-Q4"   the brackets forgotten — refused with `must be "ALL" or
#                          "OUTSTANDING"`, an error about the MODE for a mistake in the LIST.
# The narrowing sentinel is `ONLY_MISSING` now, the list is only ever a list, and ONE message
# covers a string, a `None` and anything else that is not one.
if not isinstance(QUARTERS, (list, tuple)):
    raise TypeError(f'QUARTERS is a LIST of quarters — [] or ["2014-Q4"] — not {QUARTERS!r}. '
                    f"{job.QUARTER_FORM}. An EMPTY list is every quarter the ticker files, "
                    f"and ONLY_MISSING = True narrows it to the ones still `missing`.")
QUARTERS = job.canonical_quarters(QUARTERS)
# ⚠️ AN EMPTY LIST IS RESOLVED IN §3, NOT HERE, and it is the only thing that is: §3 is where
# the statement CSVs and the PDF index are read, and neither has been opened yet.
RESOLVE_FROM_DISK = QUARTERS is None
OUTSTANDING_ONLY = RESOLVE_FROM_DISK and ONLY_MISSING

# ⚠️ THE CASCADE IS PART OF A RUN'S PROVENANCE, NOT ONLY OF ITS COST (`TSS-1`). `tesseract@200`
# is layer 4 of 55 HERE and does not exist on a Kaggle worker, so the two machines run
# DIFFERENT cascades and a local re-parse of a T4-parsed ticker can win on a layer the row on
# disk never saw. `ONNX_ONLY` makes the two the same 53. It never overrides an explicit
# `LAYERS`, and the resolved list is recorded in the run folder either way.
from web_scraper.cafef_financials import FinancialsBuilder as _FB   # noqa: E402

if ONNX_ONLY and LAYERS is None:

    LAYERS = [_l.name for _l in _FB.LAYERS if _l.name.startswith("onnx")]

# ⚠️ TWO COMBINATIONS ARE REFUSED HERE RATHER THAN DISCOVERED AFTERWARDS, and both were
# measured on real runs:
#   (a) SPAN_OPERANDS needs OVERWRITE. A span operand is by definition a quarter already
#       reading `pdf`, so at OVERWRITE=False it is dropped before any OCR and the Q4 it
#       exists to unblock stays unwritable — the run would look complete and change nothing.
#   (b) OVERWRITE + MERGE_INTO_CSV passes `force_differs=True` into the automatic per-quarter
#       merge, i.e. it lifts DIFFERS for EVERY statement of every quarter in the run. On ACB
#       (2026-08-30) that would have replaced a 33-item balance sheet with a 19-item one while
#       repairing a different statement. §9's merge is unforced; REPAIR is the scoped escape.
if SPAN_OPERANDS and not OVERWRITE:
    raise ValueError("SPAN_OPERANDS needs OVERWRITE = True — a span operand is a quarter "
                     "already reading `pdf`, and OVERWRITE=False drops it before any OCR.")
if OVERWRITE and MERGE_INTO_CSV:
    raise ValueError("OVERWRITE = True passes force_differs into the automatic merge, which "
                     "lifts DIFFERS for every statement of the run. Leave MERGE_INTO_CSV off "
                     "and use §9 (unforced, one period at a time), or REPAIR for one row.")

# The task label EVERY progress line in this notebook carries, so it is kept SHORT: it is
# repeated on every row of every table below, and a 40-character label pushes a verdict table
# off the screen to say something §4 already printed. Two quarters or fewer are named; more
# are a count.
# ⚠️ Rebuilt in §3 when the sentinel resolves: a label reading "all quarters" over a
# three-quarter run is a progress line lying about its own denominator.
LABEL = f"{EXCHANGE}_{SYMBOL}" + (
    " " + " ".join(QUARTERS) if QUARTERS and len(QUARTERS) <= 2
    else f" {len(QUARTERS)}q" if QUARTERS else "")

# ⚠️ **ONE PLAN FOR THE WHOLE NOTEBOOK, AND THEREFORE ONE PERCENTAGE.** Every line printed
# from here down leads with `xx.x%` of THE WHOLE SESSION — not of the cell you are in — in the
# one shape `utils.progress` formats and nothing else writes:
#     ` 33.7% - step 5/15 HOSE_CTG 70q - wait kernel - [ 1.5 min] RUNNING`
# Before 2026-09-04 only §5 and §6 reported at all, each with a plan of its own, so a reader
# got `33.7%` from the run cell and bare prose from the nine cells around it and had no way to
# tell a session 3 % in from one 96 % in. The three honest denominators are still named, in
# the segments (`step 5/15`, `doc 2/3`, `page 40/96`).
# ⚠️ THE OCR STEPS ARE THE ROUND TRIP'S OWN (`kgpu.runner.RUN_STAGES`) ON KAGGLE, EMBEDDED
#    HERE RATHER THAN RUN AS A SECOND PLAN — `runner.run` looks its stages up BY KEY, so
#    handing it this plan makes its six steps six steps of this notebook and keeps ONE number
#    on the line. `final=False` is what stops its closing `done()` reading as "the notebook is
#    finished" and parking every cell after it at 100 %.
if ENVIRONMENT == "KAGGLE":
    from kgpu import runner as _runner              # noqa: E402

    _OCR_STAGES = list(_runner.RUN_STAGES)          # export upload push wait download merge
else:
    # ⚠️ Weighted 100 to match `RUN_STAGES`' own total, so the OCR is the same share of the
    # notebook on both machines and the two runs' percentages mean the same thing.
    _OCR_STAGES = [("parse", "OCR the filings", 100.0)]
OCR_KEYS = [_s[0] for _s in _OCR_STAGES]
NOTEBOOK_PLAN = [
    ("setup",    "setup",                1.0),
    ("gap",      "what is left",         1.0),
    ("job",      "resolve the job",      2.0),
    ("rehearse", "rehearse worker",      3.0),
    *_OCR_STAGES,
    ("results",  "read the run folders", 1.0),
    ("refused",  "refused vs written",   1.0),
    ("upsert",   "merge into the CSVs",  5.0),
    ("landed",   "did it land",          1.0),
    ("repair",   "repair one row",       1.0),
]
# ⚠️ THE WEIGHTS ARE NOMINAL AND SAY SO. They put the OCR where it belongs — ~86 % of the
# plan — and they measure no run: a filing accepted at layer 1 is ~1 min and one that defeats
# the cascade was 33 (§6-2-noviesdecies). A weight pretending to be measured would be §5
# rule 2 wearing a progress bar.
# ⚠️ RE-RUN §2 AFTER EDITING §1: the plan's SHAPE depends on ENVIRONMENT. The number is
#    monotone by construction, so re-running a cell out of order re-prints its step at the
#    percentage already reached rather than winding the bar back.
NB = progress.Stages(NOTEBOOK_PLAN, label=LABEL, final=False)
NB.begin("setup", f"{ENVIRONMENT} — parameters validated, repo found")
with NB.capture(nested=True):
    print(f"environment : {ENVIRONMENT}")
    print(f"ticker      : {EXCHANGE}_{SYMBOL}")
    print("quarters    : " + ("OUTSTANDING — resolved in §3 from what is on disk"
                              if OUTSTANDING_ONLY else
                              f"{QUARTERS or 'ALL — every quarter this ticker files'}"))
    # ⚠️ A KNOB THAT IS NOT READ HAS TO SAY SO WHERE IT WOULD BE READ. Silence is what lets
    # a reader believe a flag they set had an effect, and this one is inert the moment the
    # list names its own quarters.
    if QUARTERS and ONLY_MISSING:
        print("            : ⚠️ ONLY_MISSING is IGNORED — it is read only when QUARTERS is "
              "empty, and this run names its quarters.")
    print(f"overwrite   : {OVERWRITE}"
          + ("" if OVERWRITE else "   (quarters already `pdf` in all three are skipped)"))
    print(f"upsert csv  : {MERGE_INTO_CSV}"
          + ("   per quarter, as each finishes" if MERGE_INTO_CSV and ENVIRONMENT == "LOCAL"
             else "   after the pull" if MERGE_INTO_CSV else ""))
    # ⚠️ A KNOB THAT IS NOT READ HAS TO SAY SO WHERE IT WOULD BE READ — the same rule
    # `ONLY_MISSING` obeys two lines up. `MERGE_EACH` is `run_batch`'s argument and nothing
    # else's, so on KAGGLE (the worker cannot reach this disk) and on the one-process path
    # (that is `MERGE_INTO_CSV`) it is inert, and silence is what would let a reader believe
    # the CSVs were being written as the run went.
    print(f"merge each  : {MERGE_EACH}"
          + ("   each quarter is upserted the moment all three of its statements are in"
             if MERGE_EACH and ENVIRONMENT == "LOCAL" and ISOLATE_DOCUMENTS else
             "   ⚠️ IGNORED — read only on LOCAL + ISOLATE_DOCUMENTS; "
             + ("KAGGLE writes on the pull" if ENVIRONMENT == "KAGGLE"
                else "the one-process path is MERGE_INTO_CSV") if MERGE_EACH else
             "   nothing reaches the CSVs until §9"))
    print(f"bootstrap   : {FORCE_EMPTY_BAND}"
          + ("   an EMPTY `sane` band is written anyway — the only way a new ticker "
             "starts" if FORCE_EMPTY_BAND else "   an EMPTY `sane` band is REFUSED"))
    print(f"isolation   : "
          + ("one process per document, VRAM floor "
             f"{VRAM_FLOOR_MB} MiB   (`GPU-1`)" if ISOLATE_DOCUMENTS and ENVIRONMENT == "LOCAL"
             else "one process for the whole run" if ENVIRONMENT == "LOCAL" else "n/a — KAGGLE"))
    print(f"cascade     : "
          + (f"{len(LAYERS)} layer(s)" if LAYERS else "the full cascade")
          + ("   onnx only — the cascade a Kaggle worker runs (`TSS-1`)"
             if ONNX_ONLY and LAYERS else ""))
    # ⚠️ **WHAT THIS CASCADE CAN RECOVER, DERIVED FROM THE LAYERS THEMSELVES.** The methods are
    # SHARED CODE — `FinancialsBuilder.LAYERS` and the `ParseLayer` flags — not notebook
    # settings, so every ticker driven from here gets all of them and there is nothing to "turn
    # on". What the readout is for is the log: a line ending `[onnx@200+noteshead]` means a
    # widening rule won that statement, and this says which rules were even reachable.
    # ⚠️ Listed from `dataclasses.fields`, never from a hand-written list — a gloss typed here
    # would be a second copy of the cascade and would be wrong the first time a flag is added.
    # `ParseLayer`'s docstring is where each one is explained and measured.
    # ⚠️ **`is_strict` IS THE LINE THAT MATTERS**: a layer reading the page AS PRINTED must
    # never run after one that widens what may be believed, so the strict reads come first and
    # a widening layer only ever judges a statement all of them refused.
    import dataclasses                                    # noqa: E402
    import textwrap                                       # noqa: E402

    from web_scraper.cafef_financials import ParseLayer   # noqa: E402

    _CASCADE = [l for l in _FB.LAYERS if LAYERS is None or l.name in set(LAYERS)]
    _WIDE = sorted(f.name for f in dataclasses.fields(ParseLayer)
                   if any(getattr(l, f.name) is True for l in _CASCADE))
    _STRICT = sum(1 for l in _CASCADE if l.is_strict)
    print(f"recoveries  : {_STRICT} strict read(s), then {len(_CASCADE) - _STRICT} widening "
          f"layer(s) carrying {len(_WIDE)} flag(s)")
    print(textwrap.fill(" ".join(_WIDE), 92, initial_indent="              ",
                        subsequent_indent="              "))
    print(f"repo        : {REPO}")
    print(f"cwd         : {Path.cwd()}")
    print(f"code        : {REPO / 'src'}"
          + (f"   ({len(_RELOADED)} cached module(s) dropped, re-imported from disk)"
             if _RELOADED else "   (first import in this kernel)"))
    # ⚠️ The percentage is a POSITION IN THE PLAN and not a fraction of the time left — a filing
    # accepted at layer 1 of 47 costs ~1 min and one that defeats the cascade cost 33. Said here,
    # once, because it is on every line below it.
    print(f"log shape   : {progress.format_line(0.337, 'task', 'sub-task', 'detail')}")
    print(f"              overall % of THIS NOTEBOOK — {len(NOTEBOOK_PLAN)} steps, the OCR worth "
          f"{100 * sum(_s[2] for _s in _OCR_STAGES) / sum(_s[2] for _s in NOTEBOOK_PLAN):.0f}%.")
    print("              A position in the plan, never a fraction of the time — a LOWER bound, "
          "so a run finishes early rather than stalling at 99 %.")
NB.end()

  0.0% - step 1/15 HOSE_GAS - setup - KAGGLE — parameters validated, repo found


  0.0% - step 1/15 HOSE_GAS - setup - environment : KAGGLE


  0.0% - step 1/15 HOSE_GAS - setup - ticker      : HOSE_GAS


  0.0% - step 1/15 HOSE_GAS - setup - quarters    : OUTSTANDING — resolved in §3 from what is on disk


  0.0% - step 1/15 HOSE_GAS - setup - overwrite   : False   (quarters already `pdf` in all three are skipped)


...

  0.0% - step 1/15 HOSE_GAS - setup - cwd         : D:\GIT\master-thesis\src\kaggle_gpu


  0.0% - step 1/15 HOSE_GAS - setup - code        : D:\GIT\master-thesis\src   (first import in this kernel)


  0.0% - step 1/15 HOSE_GAS - setup - log shape   :  33.7% - task - sub-task - detail


  0.0% - step 1/15 HOSE_GAS - setup - overall % of THIS NOTEBOOK — 15 steps, the OCR worth 86%.


  0.0% - step 1/15 HOSE_GAS - setup - A position in the plan, never a fraction of the time — a LOWER bound, so a run finishes early rather than stalling at 99 %.


## 3 · What is left — the gap on disk, and what a re-run cannot change

In [6]:
# ── WHAT IS LEFT — the gap on disk, and what a re-run cannot change ───────────
# ⚠️ THE QUESTION THIS ANSWERS IS THE ONE THAT DECIDES `QUARTERS`, and until 2026-09-02 the
# notebook could not answer it: you had to know which (quarter, statement) cells of this ticker
# still read `missing`, and the only way to find out was an ad-hoc script over the three CSVs.
#
# ⚠️ IT IS NOT A SECOND RULE. The quarters come from `documents()` through `job.plan()` — the
# same call the run makes — "already done" is `job.parsed_reports()`, which is `pdf` and nothing
# else, and a cell a past run PROVED unproducible is dropped by `settled_absences`.
#
# ⚠️ `use_data_root()` FIRST, AND IT IS LOAD-BEARING (`CWD-1`). `fin.STATEMENTS_DIR` is a
# RELATIVE default read at call time, and §2 has just `os.chdir`-ed into `src/kaggle_gpu` — so
# without this every quarter reads `absent`, which is a legitimate state for a ticker being
# bootstrapped and therefore looks like nothing is wrong.
NB.begin("gap", "the three statement CSVs and the PDF index — no OCR")
with NB.capture(nested=True):
    from web_scraper import cafef_financials as fin      # noqa: E402
    from web_scraper import pdf_ocr_batch                # noqa: E402

    job.use_data_root(REPO / "raw_data" / "cafef")
    _builder = fin.FinancialsBuilder(logger=None)

    # ⚠️ RESOLVED, NEVER DEFAULTED — and how it resolved is printed, because "read off
    # templates.csv" and "fingerprinted over the network" are not the same claim (`TPX-1`).
    [PLAN] = pdf_ocr_batch.plan_batch(
        [SYMBOL], exchange=EXCHANGE, reports_root=REPO / "reports" / "pdf_ocr",
        allow_parent=ALLOW_PARENT, span_operands=SPAN_OPERANDS, template=TEMPLATE,
        builder=_builder)
    TEMPLATE, TEMPLATE_HOW = PLAN.template, PLAN.template_how

    print(f"{PLAN.key}   template {TEMPLATE} ({TEMPLATE_HOW})   "
          f"{PLAN.filed} quarter(s) filed, {PLAN.complete} complete")
    print("")
    # ⚠️ NO FILINGS AND NOTHING OUTSTANDING PRINT THE SAME LINE OTHERWISE, and they are opposite
    # answers: one says the ticker is done, the other that nothing was ever measured (§5 rule 2).
    if not PLAN.filed:
        print("  ⚠️ this ticker files NO document `documents()` will open — an absent PDF "
              "index, or")
        print("     everything before FINANCIALS_PERIOD_MIN. Nothing here says the ticker "
              "is done.")
    elif not PLAN.quarters and not PLAN.settled:
        print("  every filed quarter reads `pdf` in all three statements. Nothing is outstanding.")
    else:
        for _q in PLAN.quarters:
            _tag = "SPAN OPERAND — re-parsed only to record `months`" if _q in PLAN.operands else \
                   "open — a re-run could still win it"
            print(f"  {_q:9} {_tag}")
        for _q, _reports in sorted(PLAN.settled.items()):
            for _r in _reports:
                print(f"  {_q:9} {_r:18} SETTLED — the filing contains no such statement")
        print("")
        print(f"  {len(PLAN.quarters)} quarter(s) with an OPEN cell "
              f"(of which {len(PLAN.operands)} are span operands), "
              f"{sum(len(v) for v in PLAN.settled.values())} SETTLED cell(s)")

    # ⚠️ A SETTLED CELL IS `missing` FOREVER, and re-running it costs the full cascade to
    # return the same word. ACB's Q2-2009 and Q3-2009 cash flows were put through all 50
    # layers FOUR times on
    # 2026-08-30 before anything recorded why: both filings are three-page `BÁO CÁO TÀI CHÍNH TÓM
    # TẮT` forms (Mẫu CBTT-03) with no cash flow statement in them at all.
    # ⚠️ AND AN EMPTY SETTLED SET IS SILENCE, NOT A CLEAN BILL: a run older than artefact schema v4
    # recorded no reason, so a cell reading "open" here may still be unwinnable and merely
    # unmeasured (§5 rule 2).
    if PLAN.settled:
        print("")
        print("  `missing` is the correct and PERMANENT answer for the SETTLED rows (§5 rule 24).")

    # ⚠️ WHICH QUARTERS THE RUN ACTUALLY TAKES — three modes, and an EMPTY `QUARTERS` is resolved
    # HERE and nowhere else. An ONLY_MISSING that resolves to nothing RAISES rather than falling
    # through: `plan()` reads an empty `quarters` as "every quarter this ticker files", so the one
    # thing a "nothing left to do" answer must not do is silently open 70 filings.
    if OUTSTANDING_ONLY:
        if not PLAN.quarters:
            raise RuntimeError(
                f"ONLY_MISSING resolved to nothing for {PLAN.key}: every filed quarter either "
                f"reads `pdf` in all three statements or is SETTLED. Name the quarters "
                f"explicitly, or set ONLY_MISSING = False, if you meant to re-parse "
                f"something anyway.")
        QUARTERS = PLAN.quarters
    elif RESOLVE_FROM_DISK:
        # ⚠️ EVERY QUARTER THE TICKER FILES — including the ones already `pdf`, which is the point:
        # this is the mode that gets a ticker to FULL coverage rather than filling its gaps. It
        # needs OVERWRITE (validated in §2) and, on this card, ISOLATE_DOCUMENTS.
        QUARTERS = [job.as_quarter(t.period) for t in
                    job.plan(_builder, EXCHANGE, SYMBOL, allow_parent=ALLOW_PARENT,
                             template=TEMPLATE)]
        PLAN.quarters = QUARTERS
    else:
        # ⚠️ `else`, not `elif QUARTERS`: an empty list is the two branches above, so everything
        # reaching here NAMES its quarters. A truthiness test would leave a fourth, silent
        # path that ran with `PLAN.quarters` still holding §3's outstanding set — a run
        # taking quarters nobody asked for, with nothing printing a difference.
        PLAN.quarters = list(QUARTERS)

    # ⚠️ The label reaches EVERY line below, so it is a count once past two quarters — §4 prints
    # the list, and repeating it on each row of a 70-quarter verdict table buys nothing.
    LABEL = f"{EXCHANGE}_{SYMBOL}" + (" " + " ".join(PLAN.quarters)
                                      if 1 <= len(PLAN.quarters) <= 2
                                      else f" {len(PLAN.quarters)}q")
    NB.task = LABEL          # the sentinel has resolved; the denominator on the line is now true
    print("")
    _MODE = "OUTSTANDING" if OUTSTANDING_ONLY else "ALL" if RESOLVE_FROM_DISK else "explicit"
    print(f'  QUARTERS = {_MODE} -> {len(PLAN.quarters)} document(s)'
          + (f": {' '.join(PLAN.quarters)}" if len(PLAN.quarters) <= 12 else
             f": {' '.join(PLAN.quarters[:6])} … {' '.join(PLAN.quarters[-3:])}"))

    # ⚠️ **THE THREE CSVs THEMSELVES, BECAUSE `MERGE_EACH` WRITES INTO THEM AS THE RUN GOES —
    # and because a MISSING file is not the obstacle a reader expects it to be.**
    # `FinancialsBuilder._write` creates the directory and the file, so "there is no CSV yet"
    # costs nothing by itself. What costs is what a missing CSV IMPLIES: no `pdf` row on disk,
    # so `seed_history` reconstructs no magnitude band, so `sane` fails open, so every
    # statement is refused, so there is still no CSV — `BND-1`, and it is a loop that only
    # FORCE_EMPTY_BAND breaks. Printed HERE, where nothing has been spent, because the
    # alternative is learning it after a whole-ticker parse (HOSE_FPT, 2026-09-04).
    # ⚠️ `_builder._existing` is `plan_merge`'s own reader, not a second one — a count taken
    # by a different reader here could disagree with the merge that follows it.
    print("")
    CSV_ON_DISK = {}
    for _report in fin.REPORTS:
        _rows = _builder._existing(EXCHANGE, SYMBOL, TEMPLATE, _report)
        CSV_ON_DISK[_report] = sum(1 for _r in _rows.values() if _r.get("source") == "pdf")
        _path = Path(fin.statement_path(TEMPLATE, _report, EXCHANGE, SYMBOL))
        print(f"  {_report:18} "
              + (f"{len(_rows):>3} row(s), {CSV_ON_DISK[_report]:>3} `pdf`   {_path.name}"
                 if _path.is_file() else
                 f"⚠️ NO FILE — {_path.name} is created by the first write that clears the "
                 f"refusals"))
    # ⚠️ **NO `pdf` ROW ANYWHERE IS THE BOOTSTRAP CASE, AND IT IS NOT THE SAME TEST AS "NO
    # FILE".** A CSV that exists holding only `missing`/`cafef` rows seeds no band either
    # (`seed_history` reads `pdf` and nothing else, §5 rule 24), so a file-existence test would
    # call such a ticker ready and every write would still be refused.
    NEEDS_BOOTSTRAP = not any(CSV_ON_DISK.values())
    if NEEDS_BOOTSTRAP:
        print("")
        print(f"  ⚠️ {PLAN.key} HAS NO `pdf` ROW ON DISK — this run BOOTSTRAPS the ticker, and")
        print("     `seed_history` has nothing to rebuild a magnitude band from, so `sane` "
              "FAILS OPEN")
        print("     on every statement of it. A quarter whose filing produces all three is "
              "written")
        print("     anyway (`BND-1` is a loop, not a guard — MERGE_EACH says why), and the "
              "three CSVs")
        print("     are CREATED by the first such write.")
        print("     ⚠️ THOSE ROWS PASS NO MAGNITUDE GUARD. Screen them by arithmetic — two "
              "statements")
        print("        agreeing on one figure, a printed subtotal closing — before quoting "
              "any of them.")
        print("        Each is printed as it is written, recorded in the run folder's `merge` "
              "block,")
        print("        and counted again in §10.")
NB.end()

  0.9% - step 2/15 HOSE_GAS - what is left - the three statement CSVs and the PDF index — no OCR


  0.9% - step 2/15 HOSE_GAS - what is left - HOSE_GAS   template corp (detect_template (CafeF fingerprint, over the network))   61 quarter(s) filed, 16 complete


  0.9% - step 2/15 HOSE_GAS - what is left - 2008-Q4   open — a re-run could still win it


  0.9% - step 2/15 HOSE_GAS - what is left - 2010-Q4   open — a re-run could still win it


  0.9% - step 2/15 HOSE_GAS - what is left - 2011-Q4   open — a re-run could still win it


...

  0.9% - step 2/15 HOSE_GAS - what is left - 45 quarter(s) with an OPEN cell (of which 0 are span operands), 0 SETTLED cell(s)


  0.9% - step 2/15 HOSE_GAS 45q - what is left - QUARTERS = OUTSTANDING -> 45 document(s): 2008-Q4 2010-Q4 2011-Q4 2012-Q1 2012-Q3 2012-Q4 … 2025-Q1 2025-Q3 2026-Q1


  0.9% - step 2/15 HOSE_GAS 45q - what is left - balance_sheet       60 row(s),  43 `pdf`   bs_HOSE_GAS.csv


  0.9% - step 2/15 HOSE_GAS 45q - what is left - income_statement    60 row(s),  54 `pdf`   is_HOSE_GAS.csv


  0.9% - step 2/15 HOSE_GAS 45q - what is left - cash_flow           60 row(s),  26 `pdf`   cf_HOSE_GAS.csv


## 4 · The plan — what would run, before anything is spent

In [8]:
# ── THE JOB — resolved and printed, before anything is spent ──────────────────
# ⚠️ Both branches end at the SAME object. `pdf_ocr.job()` writes a `JobSpec`'s fields into the
# worker notebook's parameter cell, and the worker builds the JobSpec from them — so a LOCAL run
# and a KAGGLE run of the same parameters are one procedure on two machines, not two. What
# differs is the stack, and every run records its `stack_fingerprint`.
NB.begin("job", "resolve the spec, count the ceiling — nothing is spent")
with NB.capture(nested=True):
    SPEC = CFG = PREPARED = None

    if ENVIRONMENT == "LOCAL":
        SPEC = job.JobSpec(
            exchange=EXCHANGE, symbol=SYMBOL, periods=PERIODS, quarters=PLAN.quarters,
            allow_parent=ALLOW_PARENT, overwrite=OVERWRITE, template=TEMPLATE, layers=LAYERS,
            compare_with_disk=COMPARE, merge_into_csv=MERGE_INTO_CSV,
            force_empty_band=FORCE_EMPTY_BAND,
            notes=NOTES or f"ENVIRONMENT=LOCAL overwrite={OVERWRITE}",
        )
        # ⚠️ `prepare()` resolves the data root, the models, the TEMPLATE and the document list and
        # RAISES on any of them — no OCR, no PDF. It also raises, in as many words, when every
        # quarter you asked for is already parsed and OVERWRITE is False.
        PREPARED = SPEC.prepare()
        print("\n".join(PREPARED.describe()))
        print()
        for _t in PREPARED.tasks:
            print(f"  {_t.period:<8} {_t.file[:56]:<56} "
                  f"{os.path.getsize(_t.path) / 1024 ** 2:>6.1f} MB"
                  + ("  CUMULATIVE" if _t.cumulative else ""))
        # ⚠️ THE CEILING, BEFORE ANY OF IT IS SPENT. The bill is `pages x OCR passes`, and the 49
        # layers are only 7 passes — a layer that changes only the mapping or a gate re-maps
        # a parse the page cache already holds. Both numbers are free: `page_count` opens
        # the PDF without
        # rendering a pixel, and the pass count is a property of the cascade.
        # ⚠️ It is a CEILING, loose in the honest direction: `scan` stops as soon as all three
        # statements are behind it (BID Q3-2011 reads 7 pages of 32) and the cascade stops at the
        # first layer that accepts. What it tells you is which filing would be dear IF something in
        # it cannot be read — that is the only case that pays it.
        import fitz                                       # noqa: E402
        from web_scraper.cafef_financials import ocr_key  # noqa: E402

        PASSES = len({ocr_key(_l) for _l in PREPARED.layers})
        PAGES = 0
        for _t in PREPARED.tasks:
            try:
                with fitz.open(_t.path) as _d:
                    PAGES += _d.page_count
            except Exception as _e:                       # a damaged page tree is `scan`'s problem
                print(f"  ⚠️ could not count pages of {_t.file}: {_e}")
        print("")
        print(f"  ceiling      : {PAGES} page(s) x {PASSES} OCR pass(es) = "
              f"{PAGES * PASSES:,} page-reads at most")
        print(f"                 ~{PAGES * PASSES * 0.65 / 60:.0f} min at 0.65 s/page "
              f"(onnx@200 on this laptop; the 300/400 dpi passes cost more).")
        print("                 A filing accepted at layer 1 pays ONE pass over the pages "
              "up to its last")
        print("                 statement, which is the usual case — see the run log.")

        if PREPARED.template != "bank":
            print(f"\n⚠️ CRP-1: this is a `{PREPARED.template}` filing. `C_LIABILITIES` still "
                  f"misses on corp,\n   so the balance sheet reconciles on the TRIVIAL "
                  f"`assets == resources` — true by\n   construction on any page that reads both. "
                  f"Nothing from a non-bank run may be\n   quoted as a fundamental yet.")
    else:
        from kgpu import pdf_ocr, runner                 # noqa: E402

        CFG = pdf_ocr.job(
            SYMBOL, exchange=EXCHANGE, periods=PERIODS, quarters=PLAN.quarters,
            allow_parent=ALLOW_PARENT, overwrite=OVERWRITE, template=TEMPLATE, layers=LAYERS,
            compare=COMPARE, notes=NOTES, merge_statements=MERGE_INTO_CSV,
            # ⚠️ NOT a worker parameter. The worker cannot upsert — it writes /kaggle/working and
            # exits — so this is the PULL's knob, read by `runner.merge_statements` on this
            # machine.
            force_empty_band=FORCE_EMPTY_BAND,
        )
        print("\n".join(pdf_ocr.describe(CFG)))
        print()
        # The filings this selects are the filings the WORKER will open: `plan()` runs HERE, so the
        # payload cannot diverge from the worker's own choice.
        runner.plan(CFG)
NB.end()

  1.7% - step 3/15 HOSE_GAS 45q - resolve the job - resolve the spec, count the ceiling — nothing is spent


  1.7% - step 3/15 HOSE_GAS 45q - resolve the job - job          : pdf-ocr-gas-2008-q4-2026-q1


  1.7% - step 3/15 HOSE_GAS 45q - resolve the job - kernel       : ductrung180200/mt-pdf-ocr-gas-2008-q4-2026-q1


  1.7% - step 3/15 HOSE_GAS 45q - resolve the job - dataset      : ductrung180200/mt-cafef-filings-gas-2008-q4-2026-q1


  1.7% - step 3/15 HOSE_GAS 45q - resolve the job - filings      : HOSE_GAS  periods=all  quarters=['2008-Q4', '2010-Q4', '2011-Q4', '2012-Q1', '2012-Q3', '2012-Q4', '2013-Q1', '2013-Q2', '2013-Q3', '2014-Q1', '2014-Q2', '2014-Q3', '2014-Q4', '2015-Q1', '2015-Q3', '2016-Q1', '2016-Q2', '2016-Q3', '2017-Q1', '2017-Q2', '2017-Q3', '2017-Q4', '2018-Q1', '2018-Q3', '2018-Q4', '2019-Q1', '2019-Q3', '2020-Q1', '2020-Q2', '2020-Q3', '2020-Q4', '2021-Q1', '2021-Q2', '2021-Q3', '2022-Q1', '2022-Q2', '2022-Q3', '2022-Q4', '2023-Q1', '2023-Q3', '2024-Q1', '2024-Q3', '2025-Q1', '2025-Q3', '2026-Q1']  allow_parent=True


...

  1.7% - step 3/15 HOSE_GAS 45q - resolve the job - OVERWRITE = False  # False = SKIP quarters already `pdf` in all three statements (fill the gaps). True = re-parse them. ⚠️ It also decides which filings the payload ships, and kgpu cross-checks the two.


  1.7% - step 3/15 HOSE_GAS 45q - resolve the job - TEMPLATE = 'corp'  # None = RESOLVE (templates.csv, then CafeF's fingerprint). ⚠️ Never defaults to "bank".


  1.7% - step 3/15 HOSE_GAS 45q - resolve the job - LAYERS = ['onnx@200', 'onnx@300', 'onnx@400', 'onnx@200+relax', 'onnx@300+relax', 'onnx@200+components', 'onnx@300+components', 'onnx@200+relax+components', 'onnx@200+pad6+components', 'onnx@200+pad6+relax+components', 'onnx@300+split', 'onnx@300+split+components', 'onnx@200+join+components', 'onnx@200+join+relax+components', 'onnx@200+title', 'onnx@200+title+relax', 'onnx@200+loose', 'onnx@200+loose+relax', 'onnx@400+loose', 'onnx@400+loose+relax', 'onnx@300+loose', 'onnx@300+loose+relax', 'onnx@200+title+loose', 'onnx@200+title+loose+relax', 'onnx@200+realign', 'onnx@200+realign+relax', 'onnx@300+realign', 'onnx@300+realign+relax', 'onnx@200+realign+relax+components', 'onnx@200+notes+seam', 'onnx@200+notes+seam+relax', 'onnx@300+notes+seam', 'onnx@200+notes+seam+realign', 'onnx@200+notes', 'onnx@200+tail', 'onnx@200+tail+relax', 'onnx@300+tail', 'onnx@200+tail+relax+components', 'onnx@200+unit+tail', 'onnx@200+unit+tail+relax', 'onn

  1.7% - step 3/15 HOSE_GAS 45q - resolve the job - COMPARE = True  # score every parsed cell against the statement CSV on disk


  1.7% - step 3/15 HOSE_GAS 45q - resolve the job - NOTES = 'HOSE_GAS PDF parse on a Kaggle T4 — quarters=45 (2008-Q4..2026-Q1); template=corp (stated). The WORKER writes only a run folder; the statement CSVs are upserted on THIS machine when the folder is pulled home, through pdf_ocr_merge and its four refusals, with a backup taken first. See the `merge` block of this file for what that decided. Nothing from a non-bank template may be quoted as a fundamental (CRP-1).'


## 5 · Rehearse — KAGGLE only: the worker side, locally, no quota

In [10]:
# ── STAGE + REHEARSE — KAGGLE only: the worker side, locally, no quota ────────
# ⚠️ THE PAYLOAD IS STAGED HERE, AND IT HAS TO BE: a rehearsal runs the worker against
# `.payload/<job>/`, so there is nothing to rehearse until that exists. `export` is local and
# free — it writes the zip, it does not upload; the RUN cell below re-exports and uploads, so
# nothing here commits you to anything.
# ⚠️ The rehearsal runs no OCR pass. What it proves is that the payload holds every input the
# parse reads, under BOTH of Kaggle's mount layouts, and it prints the magnitude band `sane`
# will get. AN EMPTY BAND IS THE WARNING TO STOP FOR: `sane` fails open without one, and that
# is the documented way a run writes a wrong figure (CLAUDE.md §6-2-octodecies).
# ⚠️ ONE STEP OF THE NOTEBOOK'S OWN PLAN, NOT A SECOND PLAN. This cell used to build a
# `Stages` of its own and print `50.0% - step 1/2 …` beside a run cell printing its own
# percentage of a different denominator — two bars, neither answering "how far through the
# whole thing am I?". `inside()` moves through THIS step instead, and `capture()` re-emits
# `export`'s and `rehearse`'s own output as the DETAIL of it.
if ENVIRONMENT == "KAGGLE" and REHEARSE:
    from kgpu import export, runner               # noqa: E402

    NB.begin("rehearse", "stage the payload — local, no upload, no quota")
    with NB.capture(nested=True):
        export.export(CFG)                        # -> .payload/<job>/  (no upload)
    NB.inside(0.5, "rehearse the worker — both Kaggle mount layouts")
    with NB.capture(nested=True):
        runner.rehearse(CFG)
    NB.end("rehearsed — nothing was spent")
else:
    # ⚠️ A SKIPPED STEP CLAIMS ITS WEIGHT rather than redistributing it: the plan is the plan,
    # and "we did not have to do that" is progress through it.
    NB.skip("rehearse", "REHEARSE = False" if ENVIRONMENT == "KAGGLE"
            else "LOCAL — nothing to rehearse")


  3.4% - step 4/15 HOSE_GAS 45q - rehearse worker - stage the payload — local, no upload, no quota


  3.4% - step 4/15 HOSE_GAS 45q - rehearse worker - 45 filing(s) of HOSE_GAS (Q4-2008, Q4-2010, Q4-2011, Q1-2012, Q3-2012, Q4-2012, Q1-2013, Q2-2013, Q3-2013, Q1-2014, Q2-2014, Q3-2014, Q4-2014, Q1-2015, Q3-2015, Q1-2016, Q2-2016, Q3-2016, Q1-2017, Q2-2017, Q3-2017, Q4-2017, Q1-2018, Q3-2018, Q4-2018, Q1-2019, Q3-2019, Q1-2020, Q2-2020, Q3-2020, Q4-2020, Q1-2021, Q2-2021, Q3-2021, Q1-2022, Q2-2022, Q3-2022, Q4-2022, Q1-2023, Q3-2023, Q1-2024, Q3-2024, Q1-2025, Q3-2025, Q1-2026)


  3.4% - step 4/15 HOSE_GAS 45q - rehearse worker - documents.zip: 66 files, 258.8 MB (283.1 MB uncompressed)


  3.4% - step 4/15 HOSE_GAS 45q - rehearse worker - source.zip: 72 files from src/web_scraper, src/utils


  3.4% - step 4/15 HOSE_GAS 45q - rehearse worker - staged payload -> D:\GIT\master-thesis\src\kaggle_gpu\.payload\pdf-ocr-gas-2008-q4-2026-q1  (259.5 MB total)


  4.7% - step 4/15 HOSE_GAS 45q - rehearse worker - rehearse the worker — both Kaggle mount layouts


  4.7% - step 4/15 HOSE_GAS 45q - rehearse worker - rehearsing pdf-ocr-gas-2008-q4-2026-q1 — flat mount, source.zip


  4.7% - step 4/15 HOSE_GAS 45q - rehearse worker - rehearsing pdf-ocr-gas-2008-q4-2026-q1 — datasets/ductrung180200/mt-cafef-filings-gas-2008-q4-2026-q1 mount, source/ unpacked


  4.7% - step 4/15 HOSE_GAS 45q - rehearse worker - both mount layouts pass; the built notebook is staged in D:\GIT\master-thesis\src\kaggle_gpu\.build


  6.0% - step 4/15 HOSE_GAS 45q - rehearse worker - rehearsed — nothing was spent


## 6 · Run

In [12]:
# ── RUN ───────────────────────────────────────────────────────────────────────
# ⚠️ One line shape on both machines — ` 33.7% - <task> - <sub-task> - <detail>`, one formatter
# (`utils.progress`), so the two cannot drift. LOCAL the task is the DOCUMENT and the sub-task
# its position in the cascade; KAGGLE the task is the STEP of the round trip.
# ⚠️ THE OVERALL % IS A POSITION IN THE PLAN, NOT A FRACTION OF THE TIME. A filing accepted at
# its FIRST OCR pass is ~1 min and one that defeats all 24 of them was 33, so the number is a
# LOWER BOUND — a run finishes early, it does not stall at 99 %.
#
# ⚠️ **`ISOLATE_DOCUMENTS` IS WHAT MAKES A WHOLE-TICKER RUN POSSIBLE ON THIS CARD.** Measured
# 2026-09-02: an 18-document run inside ONE process cleared three filings and then every
# `onnx@*` layer raised `CUDA failure 2: out of memory` — 294 of them — and the cascade went on
# and reported `pdf` for statements it had been unable to read. `pdf_ocr_batch.run_batch` spawns
# one process per document and waits for the card to have `VRAM_FLOOR_MB` free before each; the
# same 25 documents then ran with **0 engine errors**. It changes no semantics, because
# `seed_history` re-seeds `sane` from DISK per document and `PdfParser._ocr_cache` is scoped to
# one filing — see the module docstring.
FOLDERS: list = []
LATEST = EXIT = None

if not EXECUTE:
    # ⚠️ SKIPPED BY NAME, one line each, rather than one jump to the last OCR step. `skip`
    # advances to a stage's CEILING, so skipping the last would claim all six on a line
    # reading "merge into repo" — a step nobody asked about, credited for work nobody did.
    for _k in OCR_KEYS:
        NB.skip(_k, "EXECUTE = False — the plan above is resolved and nothing was spent")
elif ENVIRONMENT == "LOCAL" and ISOLATE_DOCUMENTS:
    from web_scraper import pdf_ocr_batch                # noqa: E402

    # ⚠️ **A BOOTSTRAP IS SAID BEFORE IT HAPPENS, NOT REFUSED.** Until 2026-09-06 this
    # raised here — a ticker with no `pdf` row has no magnitude band, so every write was
    # refused, and the run would have parsed every filing and landed none of them. The write
    # now happens for any quarter whose filing produced all three statements, so what is left
    # to do is name what that costs, once, where the hours are about to be spent.
    NB.begin("parse", f"{len(PLAN.quarters)} document(s), one process each, "
                      f"VRAM floor {VRAM_FLOOR_MB} MiB"
                      + ("   upserting each quarter as its three statements land"
                         if MERGE_EACH else ""))
    # ⚠️ AFTER `begin`, so the line carries the PARSE stage's label and its percentage — a
    # warning printed at the previous stage's position reads as being about that stage.
    if NEEDS_BOOTSTRAP and MERGE_EACH:
        NB.note(f"{PLAN.key} has no `pdf` row on disk — the three CSVs are CREATED by the "
                f"first quarter that produces all three statements, and every row this run "
                f"writes passes NO magnitude guard (`sane` fails open with no band, `BND-1`). "
                f"Screen them by arithmetic before quoting any of them.")
    # ⚠️ `progress=NB` is what stops the bar standing still through the longest thing the
    # notebook does: `run_batch` moves it ONE DOCUMENT AT A TIME through this step, and its own
    # lines come out as its detail. ⚠️ Each document is a SUBPROCESS that inherits stdout, so
    # its per-page lines go to the kernel log rather than into this cell — they are in that
    # document's own `run.log`, in the same shape.
    # ⚠️ **`merge_each` IS THE ONLY THING HERE THAT REACHES `raw_data/`.** `force_differs` is
    # not in `run_batch`'s signature and cannot be passed from it, so THREE of the four
    # refusals stand exactly as they do in §9 — a cumulative income statement whose priors it
    # cannot subtract, a figure that DIFFERS from a good `pdf` row, and a document any of
    # whose layers RAISED. What changes is WHEN: a quarter is written between documents, so an
    # interrupted run keeps what it has read.
    # ⚠️ **THE FOURTH — AN EMPTY `sane` BAND — IS LIFTED, AND THERE IS NO `force_empty_band`
    # ARGUMENT LEFT TO DECIDE OTHERWISE.** This path only ever merges a quarter whose filing
    # produced all three statements, and such a quarter is written band or no band: refusing
    # it is `BND-1`'s loop rather than a guard (§1's MERGE_EACH has the reasoning). Every such
    # row is printed as it is written, recorded in the run folder as `band: 0`, and counted
    # again in §10 — and it passed NO magnitude guard. FORCE_EMPTY_BAND still reaches §9,
    # which is what sees the quarters this path HELD.
    FOLDERS = pdf_ocr_batch.run_batch(
        [PLAN], layers=LAYERS, allow_parent=ALLOW_PARENT, overwrite=OVERWRITE,
        compare=COMPARE, notes=NOTES or f"{EXCHANGE}_{SYMBOL} — one process per document",
        vram_floor_mb=VRAM_FLOOR_MB, merge_each=MERGE_EACH, merge_apply=MERGE_APPLY,
        merge_reports=MERGE_REPORTS, progress=NB)
    NB.end(f"{len(FOLDERS)} run folder(s)")
elif ENVIRONMENT == "LOCAL":
    # ⚠️ THE OLD PATH, AND IT IS KEPT FOR ONE DOCUMENT AT A TIME. `job.run` parses every planned
    # filing in THIS process, which is right for a repair of one quarter and is what died at
    # document 4 of 18. It prints the progress line itself and writes the SAME line into the run
    # folder's `run.log`, so what you read here is what a later reader gets.
    # ⚠️ `nested=True` because those lines ALREADY lead with a percentage — of that run's own
    # documents, not of this notebook. Printed verbatim they would put two numbers on one line
    # and the second would appear to walk backwards; split, the inner percentage is dropped and
    # its `layer 12/47 …` / `page 40/96 …` segments are kept.
    NB.begin("parse", "one process for the whole run — right for ONE document")
    with NB.capture(nested=True):
        LATEST = job.run(SPEC)
    FOLDERS = [LATEST]
    NB.end(LATEST.name)
else:
    from kgpu import runner                              # noqa: E402

    if MERGE_INTO_CSV:
        NB.note("MERGE_INTO_CSV is on: accepted statements are upserted into "
                "raw_data/.../statements/ after the pull, with a backup taken first "
                "and every changed cell printed.")
    # `refresh_data=True` re-exports and re-uploads the payload every time — correct, because
    # the filter above may have changed since the last run of this job.
    # ⚠️ THE NOTEBOOK'S OWN PLAN IS HANDED STRAIGHT TO `runner.run`, because six of its stages
    # ARE the round trip's (§2 embedded `RUN_STAGES` by key). So the round trip reports as
    # steps 5..10 of 15 and the reader keeps ONE number. `final=False` (§2) is why its closing
    # `done()` ends the round trip rather than the notebook.
    EXIT = runner.run(CFG, refresh_data=True, progress=NB)
    NB.note(f"exit {EXIT}   (0 = COMPLETE and pulled)")

  6.0% - step 5/15 HOSE_GAS 45q - stage payload - pdf-ocr-gas-2008-q4-2026-q1


  6.0% - step 5/15 HOSE_GAS 45q - stage payload - 45 filing(s) of HOSE_GAS (Q4-2008, Q4-2010, Q4-2011, Q1-2012, Q3-2012, Q4-2012, Q1-2013, Q2-2013, Q3-2013, Q1-2014, Q2-2014, Q3-2014, Q4-2014, Q1-2015, Q3-2015, Q1-2016, Q2-2016, Q3-2016, Q1-2017, Q2-2017, Q3-2017, Q4-2017, Q1-2018, Q3-2018, Q4-2018, Q1-2019, Q3-2019, Q1-2020, Q2-2020, Q3-2020, Q4-2020, Q1-2021, Q2-2021, Q3-2021, Q1-2022, Q2-2022, Q3-2022, Q4-2022, Q1-2023, Q3-2023, Q1-2024, Q3-2024, Q1-2025, Q3-2025, Q1-2026)


  6.0% - step 5/15 HOSE_GAS 45q - stage payload - documents.zip: 66 files, 258.8 MB (283.1 MB uncompressed)


  6.0% - step 5/15 HOSE_GAS 45q - stage payload - source.zip: 72 files from src/web_scraper, src/utils


  6.0% - step 5/15 HOSE_GAS 45q - stage payload - staged payload -> D:\GIT\master-thesis\src\kaggle_gpu\.payload\pdf-ocr-gas-2008-q4-2026-q1  (259.5 MB total)


...

 83.6% - step 10/15 HOSE_GAS 45q - merge into repo - -> reports/pdf_ocr


 83.6% - step 10/15 HOSE_GAS 45q - merge into repo - merged 20260907-064620__hose_gas__pdf_ocr -> reports/pdf_ocr/


 83.6% - step 10/15 HOSE_GAS 45q - merge into repo - 1 run folder(s) are now in the repo's report root — `final_features` will see them like any local run.


 92.2% - step 10/15 HOSE_GAS 45q - merge into repo - COMPLETE and pulled — round trip 159.1 min


 92.2% - step 10/15 HOSE_GAS 45q - merge into repo - exit 0   (0 = COMPLETE and pulled)


## 7 · The result — verdicts from the run folder

In [14]:
# ── READ THE RUN FOLDERS ────────────────────────────────────────────────
# ⚠️ Read back from disk rather than from anything in memory, so this measures what a later
# reader would actually get. `metadata.json` already carries the whole scorecard in `results`.
# ⚠️ **A BATCH IS MANY FOLDERS, ONE PER DOCUMENT.** `FOLDERS` comes from the run cell; when the
# kernel was restarted between the two, fall back to this ticker's folders newer than the
# newest CSV backup — never to "the newest folder" alone, which on a batch is the LAST document
# and would report a 70-quarter run as a one-quarter one.
NB.begin("results", "read back from disk, not from memory")
with NB.capture(nested=True):
    import json                                          # noqa: E402

    PATTERN = f"*__{EXCHANGE.lower()}_{SYMBOL.lower()}__pdf_ocr"
    if not FOLDERS:
        FOLDERS = sorted((REPO / "reports" / "pdf_ocr").glob(PATTERN), key=lambda p: p.name)[-1:]
    LATEST = FOLDERS[-1] if FOLDERS else None
    META = MERGE = None
    RESULTS: list = []

    if LATEST is None:
        print(f"no run folder matching {PATTERN}")
    else:
        META = json.loads((LATEST / "metadata.json").read_text(encoding="utf-8"))
        inputs, ocr = META.get("inputs", {}), META.get("environment", {}).get("ocr", {})
        SCHEMA = META.get("schema_version", 1)
        print(f"{len(FOLDERS)} run folder(s), {FOLDERS[0].name} … {LATEST.name}")
        print(f"  commit       : {META.get('git_commit')}")
        print(f"  template     : {inputs.get('template')}  ({inputs.get('template_how')})")
        # ⚠️ THE TWO OCR HALVES FAIL INDEPENDENTLY — detection is onnxruntime, recognition is torch
        # — so "the GPU was used" is two questions. `ORT-1` is a green run that was half on the CPU
        # because onnxruntime ADVERTISED a provider the session then could not create.
        print(f"  detection    : {(ocr.get('det_providers') or ['?'])[0]}"
              f"   (onnxruntime {ocr.get('onnxruntime')})")
        print(f"  recognition  : {ocr.get('recognizer_device')}")
        print(f"  stack        : {ocr.get('stack_fingerprint')}"
              + (f"   ⚠️ PIN VIOLATIONS: {ocr['pin_violations']}"
                 if ocr.get("pin_violations") else ""))

        # ⚠️ **A RUN WHOSE ONNX LAYERS RAISED REPORTS `pdf` WITH A REAL LAYER AND A REAL ITEM
        # COUNT.** The rows below look identical to a good run; what happened is that the layer
        # could not run, the cascade went on, and something later won BY DEFAULT — which on this
        # machine is `tesseract@200`, layer 4 of 55. Measured 2026-09-02 on HOSE_CTG: 85 layers
        # raised `CUDA failure 2: out of memory` and 30 of 33 statements were reported `pdf`.
        # ⚠️ `pdf_ocr_merge` refuses such a document whole (`VCR-1`), so nothing reaches disk — but
        # that is the LAST line of defence and it is silent about WHY until §8. This says it here,
        # where the verdict table is read.
        RAISED = {}
        for _folder in FOLDERS:
            for _doc in sorted((_folder / "documents").glob("*.json")):
                _d = json.loads(_doc.read_text(encoding="utf-8"))
                if _d.get("engine_errors"):
                    RAISED[_d.get("period", _doc.stem)] = _d["engine_errors"]
            _m = json.loads((_folder / "metadata.json").read_text(encoding="utf-8"))
            RESULTS += _m.get("results", [])
        if RAISED:
            print()
            print(f"  ⚠️ {len(RAISED)} document(s) had at least one layer RAISE rather than "
                  f"refuse.")
            print("     Whatever won them won BY DEFAULT, and the merge refuses them whole.")
            for _p in sorted(RAISED)[:8]:
                print(f"       {_p:10} {len(RAISED[_p])} layer(s): "
                      f"{', '.join(l for l, _ in RAISED[_p][:3])}")
            _kinds = sorted({str(w).split(";")[0].strip()[:70]
                             for e in RAISED.values() for _l, w in e})
            for _k in _kinds[:3]:
                print(f"       cause: {_k}")
            print("     ⚠️ `out of memory` means the card was short — raise VRAM_FLOOR_MB, close "
                  "other")
            print("        CUDA processes, and re-run those quarters. Nothing of theirs is "
                  "on disk.")


        # ⚠️ **WHICH FILING EACH STATEMENT ACTUALLY CAME FROM (`ALT-1`).** `documents()` returns
        # ONE document per period and a quarter can have several, so a statement every layer
        # refused on the chosen filing is retried on the others of the same period and ENTITY.
        # When that succeeds the row on disk names THAT filing, not the one the document block
        # above names — and if this cell did not print it, nothing a reader sees would.
        # ⚠️ Measured on TCB Q2-2019: its closing balance is printed under the company's round
        # stamp in the AUDITED filing and no engine, DPI or crop reads it, while the REVIEWED
        # filing of the same quarter reads the whole tail cleanly at layer 1.
        ALT = {}
        for _folder in FOLDERS:
            for _doc in sorted((_folder / "documents").glob("*.json")):
                _d = json.loads(_doc.read_text(encoding="utf-8"))
                for _rep, _got in (_d.get("accepted") or {}).items():
                    if _got.get("document"):
                        ALT[(_d["period"], _rep)] = (_got["document"],
                                                     _got.get("assurance", ""))
        if ALT:
            print()
            print(f"  ⚠️ {len(ALT)} statement(s) came from a DIFFERENT filing of the same "
                  f"period and entity:")
            for (_p, _rep), (_file, _ass) in sorted(ALT.items()):
                print(f"       {_p:10} {_rep:18} {_ass:10} {_file}")
            print("     ⚠️ The ENTITY is fixed by `alternates`, so none of these changed which "
                  "company the")
            print("     row describes; the ASSURANCE may be lower, and that is the trade.")
        print()
        print(f"  {'period':10} {'report':18} {'layer':30} {'items':>5}  {'status':8} verdict")
        for r in sorted(RESULTS, key=lambda r: (fin._period_key(r["period"]), r["report"])):
            print(f"  {r['period']:10} {r['report']:18} {(r['layer'] or '—'):30} "
                  f"{r['items']:>5}  {r['status']:8} {r['verdict']}")
        # ⚠️ `seconds` is the DOCUMENT's cost repeated on each of its three report rows, so it is
        # summed per PERIOD. A set would also collapse two documents that took the same time.
        PER_DOC = {r["period"]: r["seconds"] for r in RESULTS}
        _ok = sum(1 for r in RESULTS if r["status"] == "pdf")
        print(f"\n  parse: {sum(PER_DOC.values()) / 60:.1f} min over {len(PER_DOC)} document(s)"
              f"   {_ok} of {len(RESULTS)} statement(s) accepted")
NB.end()

 92.2% - step 11/15 HOSE_GAS 45q - read the run folders - read back from disk, not from memory


 92.2% - step 11/15 HOSE_GAS 45q - read the run folders - 1 run folder(s), 20260907-064620__hose_gas__pdf_ocr … 20260907-064620__hose_gas__pdf_ocr


 92.2% - step 11/15 HOSE_GAS 45q - read the run folders - commit       : 38bc1873+dirty


 92.2% - step 11/15 HOSE_GAS 45q - read the run folders - template     : corp  (override)


 92.2% - step 11/15 HOSE_GAS 45q - read the run folders - detection    : CUDAExecutionProvider   (onnxruntime 1.22.0)


...

 92.2% - step 11/15 HOSE_GAS 45q - read the run folders - Q3-2025    income_statement   onnx@400+deskew                   17  pdf      DIFFERS


 92.2% - step 11/15 HOSE_GAS 45q - read the run folders - Q1-2026    balance_sheet      —                                  0  absent   absent in this run


 92.2% - step 11/15 HOSE_GAS 45q - read the run folders - Q1-2026    cash_flow          —                                  0  absent   absent in this run


 92.2% - step 11/15 HOSE_GAS 45q - read the run folders - Q1-2026    income_statement   —                                  0  absent   absent in this run


 92.2% - step 11/15 HOSE_GAS 45q - read the run folders - parse: 155.9 min over 45 document(s)   76 of 135 statement(s) accepted


## 8 · Refused vs written — two questions, two places

In [16]:
# ── WHAT WAS REFUSED, AND WHAT WAS WRITTEN ───────────────────────────────
#   the PARSE refused a statement   -> the document JSON's `absent_reasons`, and `run.log`
#   the MERGE refused a statement   -> the `merge` block, written by whatever ran the UPSERT
# ⚠️ ON KAGGLE THOSE ARE TWO MACHINES. A cell that greps the worker's `run.log` for
# `WRITE `/`skip ` finds nothing on a Kaggle run and, finding nothing, used to print "no
# refusals — every statement was accepted". That false success was printed over a run that
# wrote 0 of 201 accepted cells (HOSE_CTG, 2026-08-30).
#
# ⚠️ **THE REASON IS DATA NOW, NOT PROSE** (`absent_reasons`, artefact schema v4) — and since
# 2026-09-02 so are the ROWS behind it (`absent_rows`). A reason names the SYMPTOM (`no total
# assets`); the rows say WHAT THE FILING PRINTS where the chart expects that anchor, which is
# the only thing a fix can be written from. Recovering that used to cost a second OCR run.
NB.begin("refused", "the PARSE refused, and what the MERGE decided")
with NB.capture(nested=True):
    if FOLDERS:
        print("── the PARSE refused ────────────────────────────────────────")
        ABSENT: dict = {}
        for _folder in FOLDERS:
            for _doc in sorted((_folder / "documents").glob("*.json")):
                _d = json.loads(_doc.read_text(encoding="utf-8"))
                for _rep, _tried in (_d.get("absent_reasons") or {}).items():
                    ABSENT[(_d["period"], _rep)] = (
                        _tried, (_d.get("absent_rows") or {}).get(_rep))
        if not ABSENT:
            print("  nothing — every statement the cascade opened was accepted")
        for (_period, _rep), (_tried, _rows) in sorted(
                ABSENT.items(), key=lambda kv: (fin._period_key(kv[0][0]), kv[0][1])):
            print(f"  {_period:9} {_rep:18}")
            for _layer, _why in _tried:
                print(f"      [{_layer:28}] {_why}")
            # ⚠️ THE ROWS ARE THE CAUSE AND THE REASON IS THE SYMPTOM. Printed only for the
            # statements this run could not accept, and only the EARLIEST reading of them — the
            # last layer is always the most relaxed one and its rows answer a question nobody
            # asked (§6-2-duovicies' trap for the reason applies to the rows too).
            if _rows and SHOW_ABSENT_ROWS:
                print(f"      rows read at [{_rows['layer']}], pages {_rows['pages']}, "
                      f"{len(_rows['rows'])} row(s) — the ones naming a TOTAL:")
                for _r in _rows["rows"]:
                    _lab = (_r["label"] or "").upper()
                    if any(w in _lab for w in ("TỔNG", "TONG", "CUỐI", "CUOI", "ĐẦU", "DAU")):
                        print(f"        {_r['key'][:54]:54} {_r['values'][:2]}")
                        print(f"          {_r['label'][:96]}")

        # ⚠️ NOT a refusal — a fact about the FILING. A page whose scan is turned reads as vertical
        # noise, and before 2026-08-30 that cost a whole statement in silence (BID Q3-2011).
        TURNED = [ln for _f in FOLDERS
                  for ln in (_f / "run.log").read_text(encoding="utf-8",
                                                       errors="replace").splitlines()
                  if "text lines are vertical" in ln]
        if TURNED:
            print("\n── pages the READ had to turn ──────────────────────────────")
            for ln in TURNED[:12]:
                print("  " + progress.detail_of(ln))

        print("\n── the MERGE decided ───────────────────────────────────────")
        EVENTS = []
        for _folder in FOLDERS:
            _m = json.loads((_folder / "metadata.json").read_text(encoding="utf-8"))
            for _ev in (_m.get("merge") or {}).get("events", []):
                EVENTS += [(d, _ev["applied"]) for d in _ev["decisions"]]
        if EVENTS:
            for _d, _applied in sorted(EVENTS, key=lambda e: (fin._period_key(e[0]["period"]),
                                                              e[0]["report"])):
                mark = "WRITE " if _d["action"] == "write" else "skip  "
                items = f"[{_d['layer']}] {_d['items']} items" if _d["layer"] else ""
                print(f"  {mark} {_d['period']:9} {_d['report']:18} {items:32} {_d['reason']}")
                # ⚠️ A CAVEAT ON A WRITE IS LOUDER THAN A REFUSAL, because a refusal stops and a
                # write does not. Printed only for a WRITE: refusal 1 sets the note before
                # refusals 2-4 have had their say, so beside `skip` it would contradict the line.
                if _d.get("note") and _d["action"] == "write":
                    print(f"           ⚠️  {_d['note']}")
            _w = sum(1 for d, a in EVENTS if d["action"] == "write" and a)
            print(f"\n  -> {_w} statement(s) written, {len(EVENTS) - _w} refused or planned only")
            if not _w:
                print("  ⚠️ NOTHING REACHED raw_data/. On a ticker with no CSV yet the commonest "
                      "reason is an")
                print("     EMPTY `sane` band — set FORCE_EMPTY_BAND = True. It lifts ONE guard "
                      "and no other,")
                print("     so screen the artefact before quoting anything (`BND-1`).")
        else:
            print("  ⚠️ NO MERGE RAN against these run folders — the statement CSVs were not "
                  "opened.")
            print("     §9 below does it: MERGE_TWO_PASS = True, then MERGE_APPLY = True.")
NB.end()

 93.1% - step 12/15 HOSE_GAS 45q - refused vs written - the PARSE refused, and what the MERGE decided


 93.1% - step 12/15 HOSE_GAS 45q - refused vs written - ── the PARSE refused ────────────────────────────────────────


 93.1% - step 12/15 HOSE_GAS 45q - refused vs written - Q4-2008   balance_sheet


 93.1% - step 12/15 HOSE_GAS 45q - refused vs written - [onnx@200                    ] reconcile: 1 figure(s) split across two boxes — this reading is fragmented


 93.1% - step 12/15 HOSE_GAS 45q - refused vs written - [onnx@300                    ] reconcile: section sum does not close: a_tai_san_ngan_han + b_tai_san_dai_han = 16,507,862,292,018 against a printed 16,507,362,792,018


...

 93.1% - step 12/15 HOSE_GAS 45q - refused vs written - page 24: text lines are vertical (25/25 boxes) — reading it at /Rotate 90


 93.1% - step 12/15 HOSE_GAS 45q - refused vs written - page 25: text lines are vertical (21/21 boxes) — reading it at /Rotate 90


 93.1% - step 12/15 HOSE_GAS 45q - refused vs written - ── the MERGE decided ───────────────────────────────────────


 93.1% - step 12/15 HOSE_GAS 45q - refused vs written - ⚠️ NO MERGE RAN against these run folders — the statement CSVs were not opened.


 93.1% - step 12/15 HOSE_GAS 45q - refused vs written - §9 below does it: MERGE_TWO_PASS = True, then MERGE_APPLY = True.


## 9 · The merge — one period at a time, oldest first, and UNFORCED

In [18]:
# ── THE MERGE — one period at a time, oldest first, and UNFORCED ──────────
# ⚠️ WHY NOT ONE CALL OVER THE FOLDER: `merge_run` runs `plan_merge` against disk FIRST and
# `_write` afterwards, so every decision in one call is taken against the SAME disk state.
# `_quarter_priors` reads a prior's `months` from that state, so the span a Q3 records reaches
# Q4's planner only in the NEXT call. That is `SPN-1`'s dependency, and it is why a batch that
# re-parses a span operand AND the Q4 it unblocks must merge them separately, oldest first.
# ⚠️ AND NOTHING HERE LIFTS A REFUSAL. `force_differs` is not passed, so a reading that
# disagrees with disk is refused exactly as it would be by default — which is the honest
# outcome, not a failure of this cell. `REPAIR` in §11 is the scoped escape.
# ⚠️ ONE BACKUP PER TICKER, taken by the first call that actually writes.
NB.begin("upsert", f"apply={MERGE_APPLY}   one period at a time, oldest first")
with NB.capture(nested=True):
    from web_scraper import pdf_ocr_batch                  # noqa: E402

    if not MERGE_TWO_PASS:
        print("MERGE_TWO_PASS = False — nothing was merged.")
    elif not FOLDERS:
        print("no run folder to merge — run the cells above first.")
    else:
        # ⚠️ `force_empty_band` IS NOT THE WHOLE ANSWER ANY MORE, so printing it alone
        # would understate what this pass will write. `merge_batch` lifts the band refusal
        # for any quarter whose filing produced ALL THREE statements — the same rule §6's
        # per-quarter write applies, so the two writers cannot disagree about one quarter —
        # and this flag governs only what that gate does not cover.
        print(f"       force_differs=False   force_empty_band={FORCE_EMPTY_BAND}"
              "   (+ always, for a quarter with all three statements)")
        print(f"       reports={MERGE_REPORTS or 'all three'}")
        # ⚠️ WHAT THIS PASS IS FOR ONCE §6 HAS ALREADY WRITTEN. Not a second write: every
        # quarter §6 landed is re-planned against disk and comes back `identical to the row
        # already on disk`, which is a CHECK. What it picks up is what §6 HELD — a filing that
        # produced two statements of three, a document whose layers RAISED — and any quarter
        # whose span operand only landed later in the run.
        if MERGE_EACH and ENVIRONMENT == "LOCAL" and ISOLATE_DOCUMENTS:
            print("       ⚠️ MERGE_EACH already wrote every quarter whose filing produced all "
                  "three")
            print("          statements. This is the SWEEP: those come back `identical to the "
                  "row")
            print("          already on disk`, and what §6 HELD is what this writes.")
        print()
        TALLY = pdf_ocr_batch.merge_batch(
            FOLDERS, apply=MERGE_APPLY, reports=MERGE_REPORTS,
            force_empty_band=FORCE_EMPTY_BAND)
        print()
        if not MERGE_APPLY:
            print("nothing was written. Set MERGE_APPLY = True to apply the plan above.")
            print("⚠️ AND THE PLAN ABOVE UNDERSTATES IT, BY CONSTRUCTION: with nothing written, a")
            print("   later period is planned against a span the earlier one has not "
                  "recorded yet,")
            print("   and reports the refusal it always would. A dry run cannot show a "
                  "second pass")
            print("   that depends on the first.")
        elif TALLY["written"]:
            print(f"{TALLY['written']} statement(s) reached raw_data/.../statements/ — §10 reads")
            print("the CSVs themselves, which is the only place the two can be told apart.")
        elif TALLY.get("already"):
            # ⚠️ **"0 WRITTEN" IS THE NORMAL OUTCOME OF THIS PASS ONCE §6 HAS ALREADY WRITTEN,
            # AND IT MUST NOT READ AS THE ALARM BELOW.** The branch after this one is `BND-1`'s
            # siren — a run that parsed and landed nothing — and printing it over a ticker
            # whose every quarter is on disk would train a reader to ignore the one message
            # that matters. What tells them apart is `already`: a statement re-planned against
            # disk and found unchanged is a CHECK that passed, not a refusal.
            print(f"0 written, {TALLY['already']} statement(s) already on disk unchanged — "
                  f"this pass")
            print("re-planned what §6 wrote and agreed with it. That is the sweep doing its "
                  "job.")
            if TALLY["skipped"]:
                print(f"⚠️ {TALLY['skipped']} statement(s) WERE refused — §8 says which and "
                      f"why. Those are the")
                print("   quarters §6 held back: a filing that produced two statements of "
                      "three.")
        else:
            # ⚠️ "THE RUN FINISHED" AND "THE CSV CHANGED" ARE DIFFERENT FACTS, and only the
            # second was ever the point. Read back from each folder's own `merge` block — the
            # structured record `record_merge` has just written — rather than from the lines
            # above, so this reports what a later reader gets and not what this cell printed.
            import collections                                # noqa: E402

            print("⚠️ NOTHING REACHED raw_data/.../statements/. Every accepted statement was")
            print("   refused, and these are the refusals, most common first:")
            WHY = collections.Counter(
                (_d["reason"] or "").split(" — ")[0].split(" because ")[0][:64]
                for _f in FOLDERS
                for _ev in (json.loads((Path(_f) / "metadata.json").read_text(encoding="utf-8"))
                            .get("merge") or {}).get("events", []) if _ev["applied"]
                for _d in _ev["decisions"] if _d["action"] != "write")
            for _reason, _n in (WHY.most_common(6)
                                or [("(no merge block — nothing was planned)", 0)]):
                print(f"     {_n:>4}  {_reason}")
            # ⚠️ THE ONE REFUSAL THAT CLOSES ON ITSELF — and since 2026-09-06 a quarter
            # with all three statements is past it before this branch can be reached, so
            # anything left here is a filing that produced TWO of three (or none).
            if any("band" in _r for _r in WHY):
                print("   ⚠️ `sane` band EMPTY is `BND-1`, and it is a LOOP: this ticker has no")
                print("      `pdf` row on disk, so `seed_history` builds no magnitude band, so")
                print("      every statement is refused, so there is still no CSV.")
                print("      ⚠️ A QUARTER WHOSE FILING PRODUCED ALL THREE STATEMENTS IS ALREADY "
                      "PAST THIS")
                print("         — both writers lift the band for it. What is refused here "
                      "produced two")
                print("         of three, which is a judgement about THAT filing and stays "
                      "yours: §8 says")
                print("         which statement is missing and why, and FORCE_EMPTY_BAND = True "
                      "writes the")
                print("         other two anyway. It LIFTS A REAL GUARD, so screen the figures "
                      "by arithmetic")
                print("         first (two statements agreeing on one figure, a printed subtotal")
                print("         closing) before quoting any of them.")
NB.end()

 94.0% - step 13/15 HOSE_GAS 45q - merge into the CSVs - apply=True   one period at a time, oldest first


 94.0% - step 13/15 HOSE_GAS 45q - merge into the CSVs - force_differs=False   force_empty_band=False   (+ always, for a quarter with all three statements)


 94.0% - step 13/15 HOSE_GAS 45q - merge into the CSVs - reports=all three


 94.0% - step 13/15 HOSE_GAS 45q - merge into the CSVs - APPLY — 45 (ticker, period) pass(es), oldest first


 94.0% - step 13/15 HOSE_GAS 45q - merge into the CSVs - skip   Q4-2008   balance_sheet                                       absent in this run


...

 94.0% - step 13/15 HOSE_GAS 45q - merge into the CSVs - skip   Q1-2026   income_statement                                    absent in this run


 94.0% - step 13/15 HOSE_GAS 45q - merge into the CSVs - nothing to write — every statement was refused, or is already on disk unchanged


 94.0% - step 13/15 HOSE_GAS 45q - merge into the CSVs - -> 1 statement(s) written, 62 already on disk unchanged, 72 refused


 94.0% - step 13/15 HOSE_GAS 45q - merge into the CSVs - 1 statement(s) reached raw_data/.../statements/ — §10 reads


 94.0% - step 13/15 HOSE_GAS 45q - merge into the CSVs - the CSVs themselves, which is the only place the two can be told apart.


## 10 · Did it land? — the statement CSVs themselves

In [20]:
# ── DID IT LAND? — the statement CSVs themselves ───────────────────────────
# ⚠️ Everything above reports what some process DECIDED; this reads what is on disk. The two
# came apart on a Kaggle round trip that finished green, wrote a complete run folder and created
# no CSV at all (`BND-1`, HOSE_BSR and again HOSE_CTG) — and no amount of log reading would have
# said so, because the merge that refused everything ran on the other machine.
NB.begin("landed", "the statement CSVs themselves — not what a process decided")
with NB.capture(nested=True):
    import csv                                            # noqa: E402

    from web_scraper import cafef_financials as fin       # noqa: E402

    # ⚠️ `CWD-1`, AND THIS CELL WALKED STRAIGHT INTO IT. `statement_path()` reads
    # `fin.STATEMENTS_DIR` at call time and its module default is RELATIVE — while the SETUP cell
    # `os.chdir`s to `src/kaggle_gpu`, where `kgpu` stages its payload. So the first version
    # of this cell reported `NO FILE` for a ticker whose three CSVs were on disk. The path
    # is PRINTED,
    # because a directory nobody names is a directory nobody checks.
    ROOT = job.use_data_root(job.DEFAULT_DATA_ROOT)

    TPL = (META or {}).get("inputs", {}).get("template") or TEMPLATE
    # ⚠️ KEYED BY (period, REPORT), not by period. A merge writes one statement of a quarter and
    # skips another — VCB Q1-2026 wrote its income statement and cash flow while its balance sheet
    # was `identical to the row already on disk` — so a period-only set credits this run with a row
    # it deliberately left alone.
    # ⚠️ **EVERY RUN FOLDER, NOT ONE — AND THIS COLUMN NEVER APPEARED ON A BATCH UNTIL
    # 2026-09-06.** It read `MERGE`, which only the one-process path ever assigns; a batch is
    # ONE FOLDER PER DOCUMENT and both writers record into their own through
    # `pdf_ocr_merge.record_merge` (§6's per-quarter upsert and §9's sweep alike), so there
    # was nothing to read and `<- N from this run` was silently never printed. A folder
    # without `metadata.json` is an interrupted child and is skipped, exactly as §9 skips it.
    import json                                          # noqa: E402

    DECIDED = [d
               for _f in (Path(_x) for _x in FOLDERS)
               if (_f / "metadata.json").is_file()
               for ev in (json.loads((_f / "metadata.json").read_text(encoding="utf-8"))
                          .get("merge") or {}).get("events", [])
               for d in ev["decisions"] if d["action"] == "write" and ev["applied"]]
    MINE = {(d["period"], d["report"]) for d in DECIDED}
    # ⚠️ **WHICH OF THIS RUN'S ROWS `sane` NEVER JUDGED — `band == 0` and nothing else.**
    # A quarter whose filing produced all three statements is written band or no band
    # (`BND-1` is a loop, not a guard — §1's MERGE_EACH), so this is the price of that
    # decision and it has to be READABLE rather than merely taken. ⚠️ `0` is "the band was
    # empty"; a MISSING `band` key is a run folder written before 2026-09-06, which is
    # "nobody recorded it" and NOT the same claim (§5 rule 2) — so the test is `== 0`, and an
    # older artefact reports nothing here rather than reporting everything.
    UNGUARDED = sorted({(d["period"], d["report"]) for d in DECIDED if d.get("band") == 0})
    if UNGUARDED:
        print(f"⚠️ {len(UNGUARDED)} of the {len(MINE)} statement(s) this run wrote PASSED NO "
              f"MAGNITUDE GUARD:")
        print("   `seed_history` had no `pdf` row on disk to rebuild a band from, so `sane` "
              "failed open.")
        for _p, _r in UNGUARDED[:12]:
            print(f"     {_p:10} {_r}")
        if len(UNGUARDED) > 12:
            print(f"     … and {len(UNGUARDED) - 12} more")
        print("   ⚠️ SCREEN THESE BY ARITHMETIC BEFORE QUOTING ANY OF THEM — two statements "
              "agreeing on")
        print("      one figure, a printed subtotal closing. This is what `sane` would have "
              "caught:")
        print("      an OCR misread by three orders of magnitude reads like a figure "
              "(`TSS-1`'s BSR")
        print("      Q3-2019 was 361,884,738 against another layer's 361,884,738,267).")
        print("   The same list is in each run folder's `merge` block, per decision, as "
              "`band: 0`.")
        print("")
    if TPL is None:
        print("no template resolved — run the cells above first")
    else:
        print(f"{ROOT / 'financials' / 'statements' / TPL}   {EXCHANGE}_{SYMBOL}")
        print()
        ANY = False
        for _report in fin.REPORTS:
            _path = Path(fin.statement_path(TPL, _report, EXCHANGE, SYMBOL))
            if not _path.is_file():
                print(f"  {_report:18} ⚠️ NO FILE — {_path.name} does not exist")
                continue
            ANY = True
            with open(_path, encoding="utf-8-sig") as _f:
                _rows = list(csv.DictReader(_f))
            _src = {}
            for _r in _rows:
                _src[_r.get("source", "")] = _src.get(_r.get("source", ""), 0) + 1
            _mine = [_r for _r in _rows if _r.get("source") == "pdf"
                     and (_r["period"], _report) in MINE]
            print(f"  {_report:18} {len(_rows):>3} quarters   "
                  + "  ".join(f"{k}={v}" for k, v in sorted(_src.items()))
                  + (f"   <- {len(_mine)} from this run" if _mine else ""))
            # ⚠️ Rule 24: a financial statement comes from the filing PDF and from nothing else.
            # A `cafef` row is an HTML transcription and must not be in this file.
            if _src.get("cafef"):
                print(f"       ⚠️ {_src['cafef']} row(s) read `source=cafef` — an HTML "
                      f"transcription. §5 rule 24 forbids it.")
        if not ANY:
            print()
            print("  ⚠️ THIS TICKER HAS NO STATEMENT CSV AT ALL. The parse is in the run folder "
                  "and")
            print("     nothing was upserted — the MERGE section above says which refusal "
                  "stopped it.")
NB.end()

 98.3% - step 14/15 HOSE_GAS 45q - did it land - the statement CSVs themselves — not what a process decided


 98.3% - step 14/15 HOSE_GAS 45q - did it land - D:\GIT\master-thesis\raw_data\cafef\financials\statements\corp   HOSE_GAS


 98.3% - step 14/15 HOSE_GAS 45q - did it land - balance_sheet       60 quarters   missing=17  pdf=43


 98.3% - step 14/15 HOSE_GAS 45q - did it land - income_statement    60 quarters   missing=6  pdf=54


 98.3% - step 14/15 HOSE_GAS 45q - did it land - cash_flow           60 quarters   missing=33  pdf=27   <- 1 from this run


## 11 · Repair one row — scoped, deliberate, read the diff first

In [ ]:
# ── REPAIR ONE ROW — scoped, deliberate, and read the diff first ──────────
# ⚠️ THE ONLY WAY THIS NOTEBOOK OVERWRITES A GOOD-LOOKING `pdf` ROW. `pdf_ocr_merge` refuses a
# figure that DIFFERS from a `pdf` row on disk, because two runs disagreeing is not settled by
# preferring the newer one. That refusal is lifted here for the NAMED pairs only — never for the
# run — and `merge_run`'s own `periods`/`reports` filter is what scopes it.
# ⚠️ A BACKUP is taken before any write. Diff EVERY COLUMN afterwards, not the figures: three
# separate runs in this repo lost only a `publish_date` and a figures-only diff called each of
# them clean (CLAUDE.md §6-2-quatervicies, §6-2-quinvicies, §6-2-quadragies).
NB.begin("repair", f"{len(REPAIR)} scoped pair(s)   apply={REPAIR_APPLY}")
with NB.capture(nested=True):
    if LATEST is not None and REPAIR:
        from web_scraper import pdf_ocr_merge                 # noqa: E402

        HOW = "APPLY" if REPAIR_APPLY else "PLAN"
        print(f"{HOW} — {len(REPAIR)} scoped repair(s) from {LATEST.name}")
        print()
        for _period, _report in REPAIR:
            print(f"── {_period} {_report} " + "─" * 46)
            _rep = pdf_ocr_merge.merge_run(
                LATEST, apply=REPAIR_APPLY, periods=[_period], reports=[_report],
                force_differs=True, force_empty_band=FORCE_EMPTY_BAND)
            if getattr(_rep, "backup", None):
                print(f"   backup: {_rep.backup}")
        if not REPAIR_APPLY:
            print()
            print("nothing was written. Set REPAIR_APPLY = True to apply the plan above.")
    elif LATEST is not None:
        print("REPAIR is empty — no row already on disk was replaced.")
        print("  A statement this run parsed that disk already holds as `pdf` was refused as")
        print("  DIFFERS and left alone. That is the default and usually right; name the")
        print("  (quarter, statement) pair in REPAIR only once the FILING has settled which")
        print("  reading is correct.")
NB.done("end of the notebook")